# 1 · QA generation and topic discovery

Builds the volatility-oriented QA units of every earnings call (paper, Section IV-A):

1. **Data.** Downloads MAEC (transcripts with line-level audio) and EC (Qin & Yang, 2019).
2. **Chunking.** Splits each transcript into $K=\lceil N_T/100\rceil$ contiguous chunks of balanced size, with boundaries on speaker turns.
3. **QA generation.** GPT-5 (low reasoning) writes transferable questions and transcript-grounded answers, with the supporting line numbers, for every chunk.
4. **Topic discovery.** Preliminary labels from the MAEC training questions (GPT-5), `text-embedding-3-large` embeddings, MiniBatch K-Means (silhouette sweep; $k=35$), cluster names (GPT-5.2), and assignment of every question to a topic (GPT-5-mini).
5. **Outputs.** One JSON file per split with the QA units, their topics, transcript lines and audio-clip map, plus `call_map.json` (ticker and date of every call).

**Runtime:** CPU. **Estimated cost:** Costs vary with provider pricing, model version, and input length. In the original configuration, the MAEC QA-generation stage—the dominant paid step—cost approximately US$50. A reasonable historical estimate for the complete documented paid-generation workflow is approximately US$50–65, excluding optional reruns. These figures are estimates, not guaranteed current prices. The released outputs of this notebook are in the data package (`qa/`), so you only need to run it to regenerate the QA units from scratch. GPT outputs are not deterministic: a new run gives different QA units and different final numbers.

**Paper's runs:** Google Colab (CPU runtime). The notebook is designed for Google Colab; it can also be run on your own machine (README, "Where the notebooks run"). A new run does not reproduce the paper's QA units on any machine, because the GPT outputs are not deterministic; the QA units of the paper are the released ones (data package, `qa/`).

**Inputs:** the MAEC and EC archives and the KeFVP split files (downloaded by this notebook); an OpenAI API key. **Outputs:** `WORK_DIR/qa_generated/`, 9 files (the format of the data package's `qa/`); intermediate files in `WORK_DIR/pipeline_work/`. The output folder is not the `qa/` folder of the data package, so a new run does not replace the released QA units. To use the new QA units, set `QA_DIR` in notebook 2 to this folder.

### How to run

1. Give your OpenAI API key: on Colab as the secret `OPENAI_API_KEY`; elsewhere as the environment variable `OPENAI_API_KEY`, or in a `.env` file in the folder you start the notebook from.
2. Edit the **settings cell** below if needed (run mode, folders). It is the only cell to edit.
3. Run all cells.

| `RUN_MODE` | What it does | Cost |
|---|---|---|
| `"smoke"` | Tests your setup: downloads the data, builds the transcripts, the chunks and the first request file, checks your API key and its access to the four models, then **stops before the first paid request** with `SMOKE TEST PASSED`. The steps after that point (also the unpaid ones, such as the download of the split files) are not run. | none |
| `"mini"` | Tests every cell, including the paid ones, on a small number of calls: by default 60 training, 5 validation and 5 test calls of each MAEC year and 10 EC calls. The QA units and topics it writes are a test output, in their own folder. | a small fraction of the full cost; not measured |
| `"full"` | The paper's pipeline: every cell, including the paid OpenAI requests (the cells marked 💲 and `PAID`). | approximately US$50–65 (historical estimate, see above) |

Run `"smoke"` first on a new setup, then `"mini"` if you want the paid steps tested before the full run. The smoke test does not run the paid steps, so it does not test them. In `"mini"` and `"full"` mode, each paid cell sends its requests as an OpenAI batch job and waits until the job is finished (up to 24 hours each; the label embeddings are a direct request); a cell that is interrupted while it waits sends, and pays for, its requests again when it is run again.

Cell labels: **[SETUP]** folders, packages, downloads, run mode (no prompt, model or pipeline code) · **[CHECK]** verifies files, the key or results, changes nothing · **[EXPERIMENT]** the paper's code, unchanged; do not edit it when reproducing the paper's pipeline.

## 0. Setup

In [ ]:
# [SETUP] Settings: the only cell to edit. Nothing here changes the prompts, the models or the pipeline.

RUN_MODE = "full"
#   "full"  : the paper's pipeline, including the paid OpenAI requests (approximately US$50-65, a historical estimate)
#   "smoke" : tests your setup and your API key, then stops before the first paid request (no cost)
#   "mini"  : every cell, paid, on a small number of calls (cost not measured); tests the whole pipeline

WORK_DIR = None
#   The folder for the outputs and the intermediate files. None = the default:
#     on Colab : /content/drive/MyDrive/qa-grounded-volatility   (Google Drive, as in the paper's runs)
#     elsewhere: qa-grounded-volatility-work, in the folder this notebook runs in
#   Any folder works. On Colab, a folder outside /content/drive is deleted when the session ends.

QA_DIR = None       # the output: None = WORK_DIR/qa_generated  (not WORK_DIR/qa, which holds the released QA units).
#                     "mini" mode adds _mini to the folder name. A full run stops if the folder already holds QA files.
AUDIO_DIR = None    # where the MAEC and EC archives (about 24 GB) are downloaded and unpacked (more again). None = the default:
#     on Colab : /content   (the runtime disk, as in the paper's runs; downloaded again in each session)
#     elsewhere: WORK_DIR/audio   (downloaded once, reused by later runs and by notebook 2)
#   The QA files store the audio folders under AUDIO_DIR; use the same AUDIO_DIR in notebook 2.

ARCHIVE_DIR = None  # optional: a folder that is kept (on Colab: on Google Drive) for the downloaded archives.
#   None: the archives are downloaded into AUDIO_DIR; on Colab that means again in every session, and Google Drive
#         can refuse a download for up to 24 hours when a file was fetched too often.
#   A folder, for example "/content/drive/MyDrive/qa-grounded-volatility/archives": the archives are downloaded
#         there once (about 24 GB: MAEC 19.6 GB, EC 4.7 GB) and every later session unpacks them from there
#         without downloading. A free Google Drive (15 GB) is too small for this.

MINI_TRAIN_CALLS = 60   # "mini" mode: training calls of each MAEC year (the topic step needs more than 100
#                         preliminary labels; the paper's run gave about 23 per 100 training questions)
MINI_EVAL_CALLS = 5     # "mini" mode: validation calls and test calls of each MAEC year
MINI_EC_CALLS = 10      # "mini" mode: EC test calls

# These settings can also be given as environment variables (for batch jobs); a variable that is set wins over
# this cell: QAV_RUN_MODE, QAV_WORK_DIR, QAV_QA_DIR, QAV_AUDIO_DIR, QAV_ARCHIVE_DIR, QAV_MINI_TRAIN_CALLS,
# QAV_MINI_EVAL_CALLS, QAV_MINI_EC_CALLS.
# The API key: OPENAI_API_KEY. Use one AUDIO_DIR for all runs that share a WORK_DIR.

In [ ]:
# [SETUP] Environment: resolves the folders, mounts Google Drive if a folder is on it, and makes the folder of the
# intermediate files the working directory (as in the paper's runs).
import os
import shutil
import subprocess
import sys

RUN_MODE = os.environ.get("QAV_RUN_MODE") or RUN_MODE
assert RUN_MODE in ("full", "smoke", "mini"), f"RUN_MODE must be full, smoke or mini, not {RUN_MODE!r}"
MINI_TRAIN_CALLS = int(os.environ.get("QAV_MINI_TRAIN_CALLS") or MINI_TRAIN_CALLS)
MINI_EVAL_CALLS = int(os.environ.get("QAV_MINI_EVAL_CALLS") or MINI_EVAL_CALLS)
MINI_EC_CALLS = int(os.environ.get("QAV_MINI_EC_CALLS") or MINI_EC_CALLS)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# This cell can be run again: it starts from the folder the notebook was started in and from the setting.
_start_dir = globals().get("_start_dir") or os.getcwd()
if "_pipeline_dir" in globals() and WORK_DIR == _pipeline_dir:
    WORK_DIR = _work_dir_setting          # WORK_DIR was set to the folder of the intermediate files below
_work_dir_setting = WORK_DIR


def _abs(path):
    return os.path.abspath(os.path.join(_start_dir, path))


PROJECT_DIR = _abs(os.environ.get("QAV_WORK_DIR") or WORK_DIR or (
    "/content/drive/MyDrive/qa-grounded-volatility" if IN_COLAB else "qa-grounded-volatility-work"))
QA_DIR = _abs(os.environ.get("QAV_QA_DIR") or QA_DIR or os.path.join(PROJECT_DIR, "qa_generated"))
AUDIO_DIR = _abs(os.environ.get("QAV_AUDIO_DIR") or AUDIO_DIR or ("/content" if IN_COLAB else os.path.join(PROJECT_DIR, "audio")))
ARCHIVE_DIR = os.environ.get("QAV_ARCHIVE_DIR") or ARCHIVE_DIR or None
if ARCHIVE_DIR:
    ARCHIVE_DIR = _abs(ARCHIVE_DIR)
if RUN_MODE == "mini" and not QA_DIR.endswith("_mini"):
    QA_DIR += "_mini"
# intermediate files; "smoke" and "mini" never write into the folders of a full run
WORK_DIR = _pipeline_dir = os.path.join(PROJECT_DIR, "pipeline_work" if RUN_MODE == "full" else "pipeline_work_" + RUN_MODE)

if IN_COLAB and any(p.startswith("/content/drive") for p in (PROJECT_DIR, QA_DIR, AUDIO_DIR, ARCHIVE_DIR or "")):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
if ARCHIVE_DIR:
    os.makedirs(ARCHIVE_DIR, exist_ok=True)
if RUN_MODE != "smoke":
    os.makedirs(QA_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print("run mode          :", RUN_MODE + (" (from the environment variable QAV_RUN_MODE)" if os.environ.get("QAV_RUN_MODE") else ""))
print("QA files (output) :", QA_DIR if RUN_MODE != "smoke" else "none in smoke mode")
print("intermediate files:", WORK_DIR, "(the working directory)")
print("audio, transcripts:", AUDIO_DIR)
print("archives          :", ARCHIVE_DIR or "in the audio folder (ARCHIVE_DIR is not set)")
print("environment       :", "Google Colab" if IN_COLAB else "not Colab")
if RUN_MODE == "smoke":
    print("\n*** SMOKE TEST: stops before the first paid OpenAI request. ***")
if RUN_MODE == "mini":
    print("\n*** MINI RUN: the whole pipeline, with paid OpenAI requests, on a small number of calls. ***")
if IN_COLAB and not QA_DIR.startswith("/content/drive") and RUN_MODE == "full":
    print("\nWARNING: the output folder is on the Colab runtime and is deleted when the session ends.")
if RUN_MODE != "smoke" and os.path.isdir(QA_DIR) and any(n.endswith(".json") for n in os.listdir(QA_DIR)):
    if RUN_MODE == "full":
        raise RuntimeError(f"{QA_DIR} already holds QA files, and this run would replace them at its end. "
                           "Set QA_DIR to another folder, or move the files away.")
    print(f"\nNOTE: {QA_DIR} already holds QA files of an earlier mini run. This run replaces them at its end.")

# The saved order of the EC transcripts (written by a later cell in the first run) holds their full paths.
_order = os.path.join(WORK_DIR, "ec_transcript_order.json")
if os.path.exists(_order):
    import json
    with open(_order, encoding="utf-8") as f:
        _first = (json.load(f) or [""])[0]
    _first = os.path.abspath(_first)
    if not _first.startswith(os.path.join(AUDIO_DIR, "")):        # os.path.join(..., ""): also right for a drive root
        if os.path.normcase(_first).startswith(os.path.normcase(os.path.join(AUDIO_DIR, ""))):
            raise RuntimeError(f"an earlier run in {WORK_DIR} wrote AUDIO_DIR as {_first[:len(AUDIO_DIR)]}; this run has "
                               f"{AUDIO_DIR}. It is the same folder, but a later cell compares the paths letter by "
                               "letter: write AUDIO_DIR exactly as before (upper and lower case).")
        raise RuntimeError(f"an earlier run in {WORK_DIR} used another AUDIO_DIR (its EC transcripts were at "
                           f"{_first}). Use that AUDIO_DIR again, or delete {_order} if no QA units of that run are kept.")

In [ ]:
# [SETUP] Packages.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "gdown"], check=True)

import numpy as np
import requests
import time
from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv

try:
    import matplotlib  # noqa: F401
    import openai  # noqa: F401
    import pandas  # noqa: F401
    import sklearn  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Outside Colab, install the packages first: pip install -r requirements-qa.txt") from e
print("packages: ready | openai", openai.__version__)

In [ ]:
# [CHECK] The OpenAI API key, before the downloads. Changes nothing and costs nothing.
# The key is looked up as the paper's code does later: environment variable, .env file, Colab secret.
_env_file = os.path.join(_start_dir, ".env")           # also the folder the notebook was started from
if os.path.exists(_env_file):
    with open(_env_file, "rb") as f:
        _raw = f.read()
    if _raw[:2] in (bytes([255, 254]), bytes([254, 255])) or bytes([0]) in _raw:
        raise RuntimeError(f"{_env_file} is not saved as UTF-8 text (Windows PowerShell and Notepad can save it as "
                           "UTF-16). Save it again with the encoding UTF-8, or set the environment variable "
                           "OPENAI_API_KEY instead.")
    for _line in _raw.decode("utf-8-sig").splitlines():      # utf-8-sig: also a file saved as "UTF-8 with BOM"
        if _line.strip().startswith("OPENAI_API_KEY") and "=" in _line and not os.environ.get("OPENAI_API_KEY"):
            os.environ["OPENAI_API_KEY"] = _line.split("=", 1)[1].strip().strip(chr(34)).strip(chr(39))
load_dotenv()
_key = os.environ.get("OPENAI_API_KEY")
if not _key and IN_COLAB:
    from google.colab import userdata
    try:
        _key = userdata.get("OPENAI_API_KEY")
    except Exception as e:
        raise RuntimeError("no OpenAI API key: add the Colab secret OPENAI_API_KEY (key icon in the left bar) and "
                           "give this notebook access to it") from e
if not _key:
    raise RuntimeError("no OpenAI API key: set the environment variable OPENAI_API_KEY, or put a line "
                       f"OPENAI_API_KEY=... in the file {os.path.join(_start_dir, '.env')}")

from openai import OpenAI

_models = ["gpt-5", "gpt-5.2", "gpt-5-mini", "text-embedding-3-large"]   # the models this notebook calls
_no_access = []
for _m in _models:
    try:
        OpenAI(api_key=_key).models.retrieve(_m)       # a free request
    except Exception as e:
        if type(e).__name__ == "AuthenticationError":
            raise RuntimeError("OpenAI does not accept this API key; check OPENAI_API_KEY") from e
        _no_access.append(f"{_m}: {type(e).__name__}: {e}")
_models_unconfirmed = bool(_no_access)
if _no_access:
    # not fatal: a restricted key may be barred from looking models up and still be allowed to use them
    print("WARNING: could not confirm that the API key can use these models; if it cannot, the paid cells fail:")
    print("  " + "\n  ".join(_no_access))
else:
    print("OpenAI API key: found, and it can use", ", ".join(_models))

In [ ]:
# [SETUP] Download and unpack the MAEC and EC archives (transcripts and audio) into AUDIO_DIR: the same
# sources and steps as in the paper's runs, and the same cell as in notebook 2.
# Audio that is already unpacked in AUDIO_DIR is used as it is and not downloaded again. With ARCHIVE_DIR set,
# the downloaded archives are kept there and used again by later sessions, which then download nothing.
# Unpacking uses the unzip and zip commands on Linux (as on Colab) and 7-Zip on Windows.
import time

import gdown

if os.name == "nt" and len(AUDIO_DIR) > 120:
    print(f"WARNING: AUDIO_DIR is a long path ({len(AUDIO_DIR)} characters). Windows limits a file path to about 260 "
          "characters, and some audio files have long names: use a short folder such as D:/qav_audio.")
MAEC_dir = os.path.join(AUDIO_DIR, "MAEC_audio", "sp1500_earningscall")
original_data_dir = os.path.join(AUDIO_DIR, "original_dataset")
_archives = ARCHIVE_DIR or AUDIO_DIR          # where the downloaded archives are put


_WINDOWS = os.name == "nt"


def _seven_zip():
    """Windows: the 7-Zip program, which unpacks the archives there (it also reads the split EC archive)."""
    for _exe in (shutil.which("7z"), shutil.which("7z.exe"), r"C:\Program Files\7-Zip\7z.exe",
                 r"C:\Program Files (x86)\7-Zip\7z.exe"):
        if _exe and os.path.exists(_exe):
            return _exe
    raise RuntimeError("7-Zip is needed to unpack the archives on Windows and was not found. "
                       "Install it from https://www.7-zip.org and run this cell again.")


def _need_tools():
    if _WINDOWS:
        _seven_zip()
        return
    for _tool in ("zip", "unzip"):
        if shutil.which(_tool) is None:
            raise RuntimeError(f"the '{_tool}' command is needed to unpack the archives and was not found "
                               "(Linux: install the zip and unzip packages)")


def _run(cmd, cwd=None, quiet=False):
    subprocess.run(cmd, cwd=cwd, check=True, stdout=subprocess.DEVNULL if quiet else None)


def _unpack(zip_path, folder):
    """Unpacks into a temporary folder that gets its final name only when the unpack is complete."""
    _tmp = folder + ".unpacking"
    if os.path.exists(_tmp):
        shutil.rmtree(_tmp)   # left by an unpack that was interrupted
    if _WINDOWS:
        _run([_seven_zip(), "x", zip_path, "-o" + _tmp, "-y"], quiet=True)
    else:
        _run(["unzip", "-q", zip_path, "-d", _tmp])
    for _attempt in range(5):
        try:
            os.rename(_tmp, folder)
            break
        except PermissionError:           # Windows: another program (a virus scanner) can hold the new folder briefly
            if _attempt == 4:
                raise RuntimeError(f"the unpacked archive is in {_tmp}, but Windows did not allow renaming it to "
                                   f"{folder}. Close programs that use that folder, rename it by hand, and run this "
                                   "cell again.")
            time.sleep(5)


_DRIVE_HELP = ("Google Drive limits how often a file can be downloaded (the limit can last up to 24 hours); try again "
               "later. Or get the archive another way (in a browser; or open the file in Google Drive, choose 'Add "
               "shortcut to Drive' and copy it from your mounted Drive, see the README's Troubleshooting) and "
               "either unpack it into AUDIO_DIR under the folder name given above, or, for "
               f"the MAEC archive, put it in {_archives} as MAEC_audio.zip and run this cell again. To download "
               "the archives only once, set ARCHIVE_DIR in the settings cell to a folder that is kept.")

_folder = os.path.join(AUDIO_DIR, "MAEC_audio")
if os.path.exists(_folder):
    print("MAEC audio: found", _folder, "(not downloaded again)")
else:
    _need_tools()
    audio_file_id = "1m1GRCHgKn9Vz9IFMC_SpCog6uP3-gFgY"
    _zip = os.path.join(_archives, "MAEC_audio.zip")
    if os.path.exists(_zip):
        print("MAEC archive: found", _zip, "(not downloaded again)")
    else:
        try:
            gdown.download(f"https://drive.google.com/uc?id={audio_file_id}", output=_zip, quiet=False)
        except Exception as e:
            raise RuntimeError(f"the MAEC audio archive was not downloaded (target folder: {_folder}). " + _DRIVE_HELP) from e
        if not os.path.exists(_zip):
            raise RuntimeError(f"the MAEC audio archive was not downloaded (target folder: {_folder}). " + _DRIVE_HELP)
    try:
        _unpack(_zip, _folder)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"the MAEC archive {_zip} could not be unpacked; it may be incomplete or damaged. Delete "
                           "that file and run this cell again to download it again.") from e

# EC dataset (Qin & Yang 2019, ACL19 release): public Google Drive folder linked from
# github.com/GeminiLn/EarningsCall_Dataset (5 split zip parts, 4.7 GB).
if os.path.exists(original_data_dir):
    print("EC audio: found", original_data_dir, "(not downloaded again)")
else:
    _need_tools()
    folder_url = "https://drive.google.com/drive/folders/1BKCANORbcmUJKkOkBOghw6uNHPqS_az1"
    _zips = os.path.join(_archives, "original_zips")
    _complete = os.path.join(_zips, "download_complete")      # written when all parts are there
    if os.path.exists(_complete):
        print("EC archive: found", _zips, "(not downloaded again)")
    else:
        try:
            gdown.download_folder(url=folder_url, output=_zips, quiet=False)
        except Exception as e:
            raise RuntimeError(f"the EC archive was not downloaded (target folder: {original_data_dir}). " + _DRIVE_HELP) from e
        if _WINDOWS and any("ACL19_Release.zip" in files for _, _, files in os.walk(_zips)):
            open(_complete, "w").close()      # on Linux the marker is written below, once the parts are joined
    _part_dir = next((d for d, _, files in os.walk(_zips) if "ACL19_Release.zip" in files), None)
    if _part_dir is None:
        raise RuntimeError(f"the EC archive was not downloaded (target folder: {original_data_dir}). " + _DRIVE_HELP)
    try:
        if _WINDOWS:
            # 7-Zip reads the split parts (ACL19_Release.z01, ..., ACL19_Release.zip) directly
            _unpack(os.path.join(_part_dir, "ACL19_Release.zip"), original_data_dir)
        else:
            _zip = os.path.join(AUDIO_DIR, "ACL19_Release_All.zip")
            if os.path.exists(_zip):
                os.remove(_zip)
            _run(["zip", "-s", "0", "ACL19_Release.zip", "--out", _zip], cwd=_part_dir)   # joins the split parts
    except subprocess.CalledProcessError as e:
        if os.path.exists(_complete):
            os.remove(_complete)
        raise RuntimeError(f"the parts of the EC archive in {_zips} could not be joined or unpacked; a part may be "
                           "missing or incomplete. Run this cell again to download them again.") from e
    open(_complete, "w").close()
    if not _WINDOWS:
        try:
            _unpack(_zip, original_data_dir)
        except subprocess.CalledProcessError as e:
            os.remove(_complete)          # the parts are not trusted any more: the next run downloads them again
            raise RuntimeError(f"the EC archive joined from the parts in {_zips} could not be unpacked; a part may be "
                               "damaged. Run this cell again to download them again.") from e

for _expected in (MAEC_dir, os.path.join(original_data_dir, "ACL19_Release")):
    if not os.path.isdir(_expected):
        raise FileNotFoundError(f"{_expected} does not exist: the unpacked archive in AUDIO_DIR has another layout. "
                                "Delete its folder there and run this cell again.")
print("audio: ready in", AUDIO_DIR)
if ARCHIVE_DIR:
    print("archives kept in", ARCHIVE_DIR, "(MAEC_audio.zip, original_zips/): later sessions unpack them without downloading")

## 1. Transcripts

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import os
import pandas as pd

data = []

# Iterate over all subdirectories
for subdir in sorted(os.listdir(MAEC_dir)):
    subdir_path = os.path.join(MAEC_dir, subdir)

    if os.path.isdir(subdir_path):
        transcript_path = os.path.join(subdir_path, "text.txt")
        if os.path.exists(transcript_path):
            with open(transcript_path, "r", encoding="utf-8") as f:
                text = f.read()


            data.append({
                "path": transcript_path,
                "text": text,
            })

maec_transcripts_df = pd.DataFrame(data)

maec_transcripts_df.head()

In [ ]:
# [SETUP] Windows only: the transcript paths written with "/" as on Linux, which the next cell expects.
# On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    maec_transcripts_df["path"] = maec_transcripts_df["path"].str.replace(chr(92), "/", regex=False)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
def extract_ticker_date(path):
    folder_name = path.split("/")[-2]
    date, ticker = folder_name.split("_", 1)
    ticker = ticker.strip()

    return pd.Series([ticker, date])

maec_transcripts_df[["ticker", "date"]] = maec_transcripts_df["path"].apply(extract_ticker_date)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
maec_transcripts_df.to_csv('maec_transcripts.csv', index=False)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
text_file_paths = []
text_contents = []
for root, dirs, files in os.walk(original_data_dir):
    for file in files:
        if file == "TextSequence.txt":
            file_path = os.path.join(root, file)
            text_file_paths.append(file_path)
            with open(file_path, 'r', encoding='utf-8') as f:
                text_contents.append(f.read())


original_transcripts_df = pd.DataFrame({"transcript_path": text_file_paths, "transcript_text": text_contents})
original_transcripts_df.head()

In [ ]:
# [SETUP] Windows only: the transcript paths written with "/" as on Linux, which the next cell expects.
# On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    original_transcripts_df["transcript_path"] = original_transcripts_df["transcript_path"].str.replace(chr(92), "/", regex=False)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
def extract_company_date(path):
    folder_name = path.split("/")[-2]
    company, date = folder_name.split("_", 1)
    company = company.strip()
    return pd.Series([company, date])

original_transcripts_df[["company_name", "date"]] = original_transcripts_df["transcript_path"].apply(extract_company_date)

original_transcripts_df.to_csv('EC_transcripts.csv', index=False)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# EC call ids are row positions in this table. The first session saves the file order, and later sessions
# (e.g. after a disconnect) restore it, so saved QAs always attach to the same calls.
import json
_ec_order_path = os.path.join(WORK_DIR, "ec_transcript_order.json")
_paths = list(original_transcripts_df["transcript_path"])
if os.path.exists(_ec_order_path):
    with open(_ec_order_path, encoding="utf-8") as f:
        _saved = json.load(f)
    assert sorted(_saved) == sorted(_paths), "the EC files differ from the first session's; check the download"
    if _saved != _paths:
        original_transcripts_df = original_transcripts_df.set_index("transcript_path").loc[_saved].reset_index()
        original_transcripts_df.to_csv('EC_transcripts.csv', index=False)
        print("EC order restored from the first session")
else:
    with open(_ec_order_path, "w", encoding="utf-8") as f:
        json.dump(_paths, f)

## 2. Audio metadata

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import ast
import pandas as pd

maec_transcripts_df = pd.read_csv('maec_transcripts.csv')

ec_transcripts_df = pd.read_csv('EC_transcripts.csv')

ec_transcripts_df.shape

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from pathlib import Path
maec_transcripts_df['audio_path'] = maec_transcripts_df['path'].apply(lambda p: str(Path(p).parent))

ec_transcripts_df['audio_path'] = ec_transcripts_df['transcript_path'].apply(
    lambda p: str(Path(p).parent / "CEO")
)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
maec_transcripts_df.to_csv('maec_transcripts.csv', index=False)

ec_transcripts_df.to_csv('EC_transcripts.csv', index=False)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import os
import re
import pandas as pd

def annotate_transcript_with_turns(row):
    transcript_path = row['transcript_path']
    base_dir = os.path.dirname(transcript_path)
    audio_dir = os.path.join(base_dir, 'CEO')

    # If no audio folder, skip
    if not os.path.exists(audio_dir):
        print(f"No audio folder found for: {audio_dir}")
        return None, None

    # Collect audio filenames
    audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.mp3')]
    if not audio_files:
        print(f"No audio files in: {audio_dir}")
        return None, None

    # Extract (turn_number, line_in_turn) pairs
    turn_map = {}
    turn_audio_map = {}

    for f in audio_files:
        match = re.search(r'_(\d+)_(\d+)\.mp3$', f)
        if match:
            turn, line_in_turn = map(int, match.groups())
            turn_map.setdefault(turn, []).append(line_in_turn)
            turn_audio_map.setdefault(turn, []).append((line_in_turn, f))

    # Sort lines in each turn
    for turn in turn_map:
        turn_map[turn] = sorted(turn_map[turn])
        turn_audio_map[turn] = [
            fname for _, fname in sorted(turn_audio_map[turn], key=lambda x: x[0])
        ]

    # Sort original (possibly non-continuous) turns
    sorted_turns = sorted(turn_map.keys())

    # Create continuous numbering: 1, 2, 3, ... in sorted order
    renumber = {old_t: new_t for new_t, old_t in enumerate(sorted_turns, start=1)}

    # Apply renumbering to audio map
    continuous_turn_audio_map = {
        renumber[old_t]: files
        for old_t, files in turn_audio_map.items()
    }

    # Read transcript lines
    try:
        with open(transcript_path, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
    except Exception as e:
        print(f"Failed to read transcript: {transcript_path}, error: {e}")
        return None, None

    # Annotate transcript lines
    annotated_lines = []
    line_idx = 0

    for old_t in sorted_turns:
        new_t = renumber[old_t]
        line_count = len(turn_map[old_t])

        for _ in range(line_count):
            if line_idx < len(lines):
                annotated_lines.append(f"[TURN {new_t}] {lines[line_idx]}")
                line_idx += 1

    # Add remaining lines (if transcript longer)
    while line_idx < len(lines):
        annotated_lines.append(f"[TURN ?] {lines[line_idx]}")
        line_idx += 1

    annotated_text = "\n".join(annotated_lines)

    return annotated_text, continuous_turn_audio_map



annotations = ec_transcripts_df.apply(annotate_transcript_with_turns, axis=1, result_type='expand')
ec_transcripts_df['annotated_transcript_text'] = annotations[0]
ec_transcripts_df['turn_audio_map'] = annotations[1]

ec_transcripts_df.to_csv('EC_transcripts.csv', index=False)

print("\n===== Sample Annotated Transcript =====")
print(ec_transcripts_df.iloc[0]['annotated_transcript_text'])
print("\n===== Sample Turn → Audio Map =====")
print(ec_transcripts_df.iloc[0]['turn_audio_map'])

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import os
import re
import pandas as pd

def annotate_maec_transcript_with_turns(row):
    transcript_path = row['path']
    audio_dir = os.path.dirname(transcript_path)

    # If no audio folder, skip
    if not os.path.exists(audio_dir):
        print(f"No audio folder found for: {audio_dir}")
        return None, None

    # Collect audio filenames
    audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.mp3')]
    if not audio_files:
        print(f"No audio files in: {audio_dir}")
        return None, None

    # Extract (turn_number, line_in_turn) pairs
    turn_map = {}        # turn → list of line numbers
    turn_audio_map = {}  # turn → list of unordered (line_in_turn, filename)

    for f in audio_files:
        # Example: ESE_20160503_f000031101.mp3
        match = re.search(r'_f0*([0-9]{3})([0-9]{3})\.mp3$', f)
        if match:
            turn = int(match.group(1))       # '031' → 31
            line_code = int(match.group(2))  # '101' → 101
            line_in_turn = (line_code - 100) + 1

            turn_map.setdefault(turn, []).append(line_in_turn)
            turn_audio_map.setdefault(turn, []).append((line_in_turn, f))

    if not turn_map:
        print(f"No valid turn info: {audio_dir}")
        return None, None

    # Sort lines within each turn
    for turn in turn_map:
        turn_map[turn] = sorted(turn_map[turn])
        turn_audio_map[turn] = [
            fname for _, fname in sorted(turn_audio_map[turn], key=lambda x: x[0])
        ]

    # Sort turns
    sorted_turns = sorted(turn_map.keys())

    # Create continuous turn numbering
    renumber = {old_t: new_t for new_t, old_t in enumerate(sorted_turns, start=1)}

    # Renumber the audio map keys
    continuous_turn_audio_map = {
        renumber[old_t]: files
        for old_t, files in turn_audio_map.items()
    }

    # Read transcript lines
    try:
        with open(transcript_path, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
    except Exception as e:
        print(f"Failed to read transcript: {transcript_path}, error: {e}")
        return None, None

    # Annotate transcript lines using new turn ids
    annotated_lines = []
    line_idx = 0

    for old_t in sorted_turns:
        new_t = renumber[old_t]
        line_count = len(turn_map[old_t])

        for _ in range(line_count):
            if line_idx < len(lines):
                annotated_lines.append(f"[TURN {new_t}] {lines[line_idx]}")
                line_idx += 1

    # Leftover lines
    while line_idx < len(lines):
        annotated_lines.append(f"[TURN ?] {lines[line_idx]}")
        line_idx += 1

    annotated_text = "\n".join(annotated_lines)

    return annotated_text, continuous_turn_audio_map

annotations = maec_transcripts_df.apply(
    annotate_maec_transcript_with_turns,
    axis=1,
    result_type='expand'
)

maec_transcripts_df['annotated_transcript_text'] = annotations[0]
maec_transcripts_df['turn_audio_map'] = annotations[1]

maec_transcripts_df.to_csv(
    'maec_transcripts.csv',
    index=False
)

## 3. QA generation

### MAEC: chunking

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import pandas as pd

maec_transcripts_df = pd.read_csv('maec_transcripts.csv')

import ast

maec_transcripts_df['turn_audio_map'] = maec_transcripts_df['turn_audio_map'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


maec_transcripts_df.shape

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
def linear_partition(seq, k):
    """
    Partition seq (list of positive ints) into k contiguous partitions
    minimizing the maximum partition sum. Returns list of partitions (lists).
    Standard DP solution.
    """
    n = len(seq)
    if k <= 0:
        return []
    if k >= n:
        # each element in its own partition (or empty partitions appended)
        return [[x] for x in seq] + [[] for _ in range(k - n)]

    # prefix sums
    prefix = [0] * (n + 1)
    for i in range(1, n + 1):
        prefix[i] = prefix[i-1] + seq[i-1]

    # DP tables
    M = [[0] * (k + 1) for _ in range(n + 1)]
    D = [[0] * (k + 1) for _ in range(n + 1)]

    # base cases
    for i in range(1, n + 1):
        M[i][1] = prefix[i]
    for j in range(1, k + 1):
        M[1][j] = seq[0]

    # fill DP
    for i in range(2, n + 1):
        for j in range(2, k + 1):
            best = None
            best_x = None
            # try partition point x between 1..i-1
            for x in range(1, i):
                cost = max(M[x][j-1], prefix[i] - prefix[x])
                if best is None or cost < best:
                    best = cost
                    best_x = x
            M[i][j] = best
            D[i][j] = best_x

    # reconstruct partitions
    partitions = []
    def reconstruct(n, k):
        if k == 1:
            partitions.append(seq[:n])
        else:
            x = D[n][k]
            reconstruct(x, k-1)
            partitions.append(seq[x:n])
    reconstruct(n, k)
    return partitions




import re
import math



# pattern to capture each transcript line that starts with [TURN X]
LINE_PATTERN = re.compile(r"^\[TURN\s+(\d+)\]\s*(.*)$", re.MULTILINE)

def split_ecc_into_chunks(transcript_text, turn_audio_map, max_lines=100):
    """
    Parse transcript into turns with aligned audio files,
    then divide into balanced chunks using linear partition,
    ensuring turns are not split.
    """

    # Extract turns (with aligned audio)
    matches = LINE_PATTERN.findall(transcript_text)
    if not matches:
        return []

    turns = []
    current_tid = None
    current_lines = []

    for tid_str, text in matches:
        tid = int(tid_str)
        full_line = f"[TURN {tid}] {text.strip()}"

        if tid == current_tid:
            current_lines.append(full_line)
        else:
            # finalize previous turn
            if current_tid is not None:
                audio_files = turn_audio_map.get(current_tid, [])
                if len(audio_files) >= len(current_lines):
                    audio_files = audio_files[:len(current_lines)]
                else:
                    audio_files = audio_files + [None] * (len(current_lines) - len(audio_files))

                turns.append({
                    "turn_id": current_tid,
                    "lines": current_lines,
                    "audio_files": audio_files
                })

            current_tid = tid
            current_lines = [full_line]

    # finalize last turn
    if current_tid is not None:
        audio_files = turn_audio_map.get(current_tid, [])
        if len(audio_files) >= len(current_lines):
            audio_files = audio_files[:len(current_lines)]
        else:
            audio_files = audio_files + [None] * (len(current_lines) - len(audio_files))

        turns.append({
            "turn_id": current_tid,
            "lines": current_lines,
            "audio_files": audio_files
        })

    # Compute target number of chunks
    total_lines = sum(len(t["lines"]) for t in turns)
    num_chunks = max(1, math.ceil(total_lines / max_lines))

    # Linear partition on turn sizes
    turn_sizes = [len(t["lines"]) for t in turns]   # e.g. [5, 17, 9, 33, ...]

    partitions = linear_partition(turn_sizes, num_chunks)
    # partitions is list of lists, each a sequence of turn sizes

    # Build final chunks
    final_chunks = []
    turn_index = 0

    for part in partitions:
        part_turns = turns[turn_index : turn_index + len(part)]
        turn_index += len(part)

        chunk_lines = []
        chunk_audio_map = {}
        line_number = 1

        for turn in part_turns:
            for i, line in enumerate(turn["lines"]):
                chunk_lines.append(line)
                if i < len(turn["audio_files"]):
                    chunk_audio_map[line_number] = turn["audio_files"][i]
                line_number += 1

        final_chunks.append({
            "text_lines": chunk_lines,
            "audio_map": chunk_audio_map
        })

    return final_chunks

In [ ]:
# [SETUP] "mini" mode: keeps a small number of MAEC calls of every split. In the other modes this cell changes nothing.
# The calls are renumbered 0, 1, 2, ... so the call ids of a mini run are not those of the paper.
if RUN_MODE == "mini":
    import urllib.request

    _KEFVP = "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec"
    _df = maec_transcripts_df
    _date = pd.to_datetime(_df["date"].astype(str), format="%Y%m%d")
    _usable = _df["annotated_transcript_text"].notna()
    _keep = []
    for _year in ("15", "16"):
        for _split, _n in (("train", MINI_TRAIN_CALLS), ("dev", MINI_EVAL_CALLS), ("test", MINI_EVAL_CALLS)):
            _name = f"maec{_year}_{_split}_avg_val.csv"      # the split files of KeFVP, as used later in this notebook
            urllib.request.urlretrieve(f"{_KEFVP}/{_year}/{_name}", _name)
            _s = pd.read_csv(_name)
            _keys = set(zip(_s["ticker"], pd.to_datetime(_s["time"])))
            _rows = [i for i in _df.index[_usable] if (_df.at[i, "ticker"], _date[i]) in _keys][:_n]
            assert len(_rows) == _n, f"only {len(_rows)} calls found for maec{_year} {_split}; lower the MINI_ settings"
            _keep += _rows
    maec_transcripts_df = _df.loc[sorted(set(_keep))].reset_index(drop=True)
    print(f"*** MINI RUN: {len(maec_transcripts_df)} of {len(_df)} MAEC calls kept "
          f"({MINI_TRAIN_CALLS} training, {MINI_EVAL_CALLS} validation, {MINI_EVAL_CALLS} test calls of each year) ***")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
all_ecc_chunks = []

for idx, row in maec_transcripts_df.iterrows():
    transcript = row["annotated_transcript_text"]
    turn_audio_map = row["turn_audio_map"]

    chunks = split_ecc_into_chunks(transcript, turn_audio_map, max_lines=100)
    all_ecc_chunks.append({
        "ecc_id": idx,
        "chunks": chunks
    })

### MAEC: QA requests

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

BATCH_FILE = "batch.jsonl"


template_developer_prompt = """
System: System: You are a financial analysis assistant.

Your role is to extract Q&A pairs from an Earnings Conference Call (ECC) excerpt.

--------------------------------------------
## REQUIREMENTS
--------------------------------------------
### QUESTION
- Formulate a general financial concept question that could influence stock volatility.
- Requirements:
  - Be abstract, high-level, concise, and focus on a single theme.
  - Avoid multi-part or combined questions.
  - Exclude:
    - Company names
    - Product names
    - Unique technologies
    - Customer identifiers
    - Any specifics (these belong only in answers)
  - Ensure questions remain conceptual and generalized.

### ANSWER
- Answers must be strictly grounded in the ECC excerpt using explicit details:
  - Include numbers, timeframes, or operational commentary as provided.
  - Do not fabricate, infer, or paraphrase beyond the text.
  - Directly address the generalized question.
- Append to each answer:
  source_lines: [start_line, end_line]

### EXCLUSIONS
- Omit greetings, thank-you statements, or introductions.
- Exclude legal disclaimers, operator prompts, procedural text, and irrelevant minutiae not affecting stock volatility.
- Entity names are prohibited in questions; include them in answers only if present in the excerpt.

--------------------------------------------
## OUTPUT FORMAT
--------------------------------------------
For each Q&A pair, use the format:

Q: <generalized, conceptual question>
A: <specific, evidence-based answer from the excerpt>
source_lines: [start_line, end_line]

If the excerpt contains no volatility-relevant content, output exactly:

[]
""".strip()



with open(BATCH_FILE, "w", encoding="utf-8") as f:

    for ecc in all_ecc_chunks:
        ecc_id = ecc["ecc_id"]


        for chunk_id, chunk in enumerate(ecc["chunks"]):

            # Build numbered chunk
            numbered_chunk = "\n".join(
                f"{idx}: {line}" for idx, line in enumerate(chunk["text_lines"], start=1)
            )

            # Refined USER prompt

            user_prompt = f"""
            Here is an ECC excerpt with numbered lines:

            {numbered_chunk}

            Apply the instructions from the developer message.
            Return the output in strict JSON format as specified earlier.
            If the excerpt contains nothing relevant to stock volatility, return [].
            """.strip()

            body = {
                "model": "gpt-5",
                "reasoning": {"effort": "low"},
                "instructions": template_developer_prompt,
                "input": user_prompt
            }

            # Responses endpoint
            entry = {
                "custom_id": f"{ecc_id}-{chunk_id}",
                "method": "POST",
                "url": "/v1/responses",
                "body": body
            }

            f.write(json.dumps(entry, ensure_ascii=False) + "\n")


print(f"Batch JSONL created: {BATCH_FILE}")

In [ ]:
# [CHECK] "smoke" mode ends here, before the first paid request. In "full" mode this cell changes nothing.
class SmokeTestPassed(Exception):
    """Raised on purpose, to stop "Run all" before the paid cells."""


if RUN_MODE == "smoke":
    with open(BATCH_FILE, encoding="utf-8") as f:
        _requests = sum(1 for _ in f)
    _calls = len(all_ecc_chunks)
    _chunks = sum(len(e["chunks"]) for e in all_ecc_chunks)
    assert _calls > 0 and _requests == _chunks > 0, "the request file does not hold one request per chunk"
    print("SMOKE TEST PASSED: every step before the first paid request ran on this setup.")
    print(f"  MAEC calls: {_calls:,} | chunks: {_chunks:,} | requests prepared: {_requests:,} | EC transcripts: {len(ec_transcripts_df):,}")
    print("  The OpenAI API key was accepted." if not _models_unconfirmed else
          "  WARNING: the API key was found, but its access to the models could NOT be confirmed (see the key check above).")
    print("  The paid steps were not run and are not tested.")
    print(f"  Test files: {WORK_DIR} (they can be deleted)")
    print('  Next: set RUN_MODE = "full" in the settings cell and run all cells again.')
    raise SmokeTestPassed('stopped here on purpose. This is not an error: see "SMOKE TEST PASSED" above.')

### 💲 MAEC QA generation (GPT-5, about 5,700 requests, about US$50)

This step sends a request batch to the OpenAI Batch API and waits until it finishes (up to 24 hours). It needs an `OPENAI_API_KEY` (Colab secret or `.env`).

In [ ]:
# [SETUP] Windows only: the request files written with the line endings of Linux, so that the next cell uploads
# the same bytes as on Colab. On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    import glob
    _crlf, _lf = bytes([13, 10]), bytes([10])
    for _file in glob.glob("*.jsonl"):
        with open(_file, "rb") as f:
            _bytes = f.read()
        if _crlf in _bytes:
            with open(_file, "wb") as f:
                f.write(_bytes.replace(_crlf, _lf))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import json
import time
from openai import OpenAI

#    UPLOAD batch file

upload = client.files.create(
    file=open(BATCH_FILE, "rb"),
    purpose="batch"
)

file_id = upload.id
print("Uploaded batch file:", file_id)

while True:
    status = client.files.retrieve(file_id).status
    print("File status:", status)
    if status == "processed":
        break
    time.sleep(3)



#   CREATE BATCH JOB
#

batch = client.batches.create(
    input_file_id=file_id,
    endpoint="/v1/responses",
    completion_window="24h"
)

batch_id = batch.id
print("Batch job created:", batch_id)

while True:
    info = client.batches.retrieve(batch_id)
    print(
        f"Batch status: {info.status} "
        f"({info.request_counts.completed}/{info.request_counts.total})"
    )

    if info.status == "completed":
        break
    if info.status in ("failed", "expired"):
        raise RuntimeError("Batch job failed or expired.")
    time.sleep(30)



#   DOWNLOAD AND PARSE RESULTS


if info.request_counts.failed:
    print(f"WARNING: {info.request_counts.failed} requests failed; error file: {info.error_file_id}")
output_file_id = info.output_file_id
result_file = client.files.content(output_file_id)
result_text = result_file.text

qa_results = {}
prompt_tokens_list = {}
completion_tokens_list = {}
reasoning_tokens_list = {}
total_tokens_list = {}

for line in result_text.splitlines():
    obj = json.loads(line)
    cid = obj["custom_id"]

    response = obj["response"]["body"]

    assistant_text = None
    for item in response.get("output", []):
        if item.get("type") == "message":
            content_list = item.get("content", [])
            if content_list and "text" in content_list[0]:
                assistant_text = content_list[0]["text"].strip()
            break

    qa_results[cid] = assistant_text

    usage = response.get("usage", {})
    prompt_tokens = usage.get("input_tokens", 0)
    completion_tokens = usage.get("output_tokens", 0)
    reasoning_tokens = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)
    total_tokens = prompt_tokens + completion_tokens

    prompt_tokens_list[cid] = prompt_tokens
    completion_tokens_list[cid] = completion_tokens
    reasoning_tokens_list[cid] = reasoning_tokens
    total_tokens_list[cid] = total_tokens

    print(
        f"ID {cid}: prompt={prompt_tokens}, completion={completion_tokens}, "
        f"reasoning={reasoning_tokens}, total={total_tokens}"
    )



first_key = list(qa_results.keys())[0]
print("\nSample output for ID =", first_key)
print(qa_results[first_key])

### MAEC: parse the answers

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

with open("maec_raw_chunks_qas_results.json", "w", encoding="utf-8") as f:
    json.dump(qa_results, f, ensure_ascii=False, indent=2)


with open("maec_raw_chunks_qas_results.json", "r", encoding="utf-8") as f:
    qas = json.load(f)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import ast
import re

def normalize_qa_dict(d):
    """Normalize QA dict keys to exactly Q / A / source_lines"""
    q = (
        d.get("Q")
        or d.get("q")
        or d.get("question")
        or d.get("Question")
    )
    a = (
        d.get("A")
        or d.get("a")
        or d.get("answer")
        or d.get("Answer")
    )

    if not q or not a:
        return None

    out = {"Q": q.strip(), "A": a.strip()}

    if "source_lines" in d:
        out["source_lines"] = d["source_lines"]

    return out


def parse_qa_value(qa_key, qa_val):
    # Case 1: already a list
    if isinstance(qa_val, list):
        qas = []
        for d in qa_val:
            if isinstance(d, dict):
                norm = normalize_qa_dict(d)
                if norm:
                    qas.append(norm)
                else:
                    print(f"[WARN] {qa_key}: dropped malformed QA → {d}")
        return qas

    # Case 2: not a string
    if not isinstance(qa_val, str):
        print(f"[WARN] {qa_key}: unsupported type {type(qa_val)}")
        return []

    # Normalize text
    text = qa_val.replace("\\$", "$").strip()

    # Case 3: list-like string
    if text.startswith("["):
        try:
            raw = ast.literal_eval(text)
            qas = []
            for d in raw:
                if isinstance(d, dict):
                    norm = normalize_qa_dict(d)
                    if norm:
                        qas.append(norm)
                    else:
                        print(f"[WARN] {qa_key}: dropped malformed QA → {d}")
            return qas
        except Exception as e:
            print(f"[WARN] {qa_key}: literal_eval failed → {e}")
            return []

    qas = []
    current = None
    mode = None  # "Q" or "A"

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        lower = line.lower()

        # Question markers
        if lower.startswith(("q:", "question:")):
            current = {"Q": line.split(":", 1)[1].strip()}
            mode = "Q"

        # Answer markers
        elif lower.startswith(("a:", "answer:")):
            if not current:
                print(f"[WARN] {qa_key}: A without Q")
                continue
            current["A"] = line.split(":", 1)[1].strip()
            mode = "A"

        # source_lines
        elif lower.startswith("source_lines"):
            if not current:
                print(f"[WARN] {qa_key}: source_lines without QA")
                continue

            nums = re.findall(r"\d+", line)
            if len(nums) >= 2:
                current["source_lines"] = [int(nums[0]), int(nums[1])]
            else:
                print(f"[WARN] {qa_key}: bad source_lines → {line}")

            if "Q" in current and "A" in current:
                qas.append(current)
            else:
                print(f"[WARN] {qa_key}: incomplete QA → {current}")

            current = None
            mode = None

        # Multiline continuation
        else:
            if current and mode:
                current[mode] += " " + line

    if current:
        print(f"[WARN] {qa_key}: dangling QA at end → {current}")

    return qas

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
total_qas = 0

ecc_qas = dict()

for qa_key, qa_val in qas.items():
    qa_list = parse_qa_value(qa_key, qa_val)
    ecc_qas[qa_key] = qa_list
    total_qas += len(qa_list)

print("Total number of QAs:", total_qas)

for k, v in ecc_qas.items():
    for qa_dict in v:
        if 'Q' not in qa_dict or 'A' not in qa_dict:
            print(f"Key: {k}, Problematic QA dictionary: {qa_dict.keys()}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

with open("maec_clean_chunks_qas_results.json", "w", encoding="utf-8") as f:
    json.dump(ecc_qas, f, ensure_ascii=False, indent=2)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
for key, qas in ecc_qas.items():
    ecc_id_str, chunk_id_str = key.split("-")
    ecc_id = int(ecc_id_str)
    chunk_id = int(chunk_id_str)
    all_ecc_chunks[ecc_id]['chunks'][chunk_id]['qas'] = qas

import json

with open("maec_ecc_chunks_with_qas.json", "w", encoding="utf-8") as f:
    json.dump(all_ecc_chunks, f, ensure_ascii=False, indent=2)

### EC (KeFVP test set): chunking and QA requests

In [ ]:
# [EXPERIMENT] The paper's code; only the download command changed (Python instead of wget, same URLs).
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/test_split_Avg_Series_WITH_LOG.csv",
    "ec_test_avg_val.csv")

import pandas as pd

ec_test_avg_val_df = pd.read_csv('ec_test_avg_val.csv')

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# Make both date columns strings
ec_test_avg_val_df = ec_test_avg_val_df.copy()

ec_test_avg_val_df["date"] = pd.to_datetime(
    ec_test_avg_val_df[["year", "month", "day"]]
).dt.strftime("%Y%m%d")

ec_transcripts_df = ec_transcripts_df.copy()
ec_transcripts_df["date"] = ec_transcripts_df["date"].astype(str)

test_keys = (
    ec_test_avg_val_df[["name", "date"]]
    .rename(columns={"name": "company_name"})
    .drop_duplicates()
)

ec_transcripts_test_df = ec_transcripts_df.merge(
    test_keys,
    on=["company_name", "date"],
    how="inner"
)

print(len(ec_transcripts_test_df))

In [ ]:
# [SETUP] "mini" mode: keeps the first EC test calls. In the other modes this cell changes nothing.
if RUN_MODE == "mini":
    _n_all = len(ec_transcripts_test_df)
    ec_transcripts_test_df = (ec_transcripts_test_df[ec_transcripts_test_df["annotated_transcript_text"].notna()]
                              .iloc[:MINI_EC_CALLS].reset_index(drop=True))
    print(f"*** MINI RUN: {len(ec_transcripts_test_df)} of {_n_all} EC test calls kept ***")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
########################################################################################
########################################################################################
########################################################################################
#### Remember that the idx refers to ecc index in ec_transcripts_test_df (not ec_transcripts_df)
########################################################################################
########################################################################################
########################################################################################
all_ecc_chunks = []

for idx, row in ec_transcripts_test_df.iterrows():
    transcript = row["annotated_transcript_text"]
    turn_audio_map = row["turn_audio_map"]

    chunks = split_ecc_into_chunks(transcript, turn_audio_map, max_lines=100)
    all_ecc_chunks.append({
        "ecc_id": idx,
        "chunks": chunks
    })

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
BATCH_FILE = "batch.jsonl"


template_developer_prompt = """
System: You are a financial analysis assistant.

Your role is to extract Q&A pairs from an Earnings Conference Call (ECC) excerpt.

--------------------------------------------
## REQUIREMENTS
--------------------------------------------
### QUESTION
- Formulate a general financial concept question that could influence stock volatility.
- Requirements:
  - Be abstract, high-level, concise, and focus on a single theme.
  - Avoid multi-part or combined questions.
  - Exclude:
    - Company names
    - Product names
    - Unique technologies
    - Customer identifiers
    - Any specifics (these belong only in answers)
  - Ensure questions remain conceptual and generalized.

### ANSWER
- Answers must be strictly grounded in the ECC excerpt using explicit details:
  - Include numbers, timeframes, or operational commentary as provided.
  - Do not fabricate, infer, or paraphrase beyond the text.
  - Directly address the generalized question.
- Append to each answer:
  source_lines: [start_line, end_line]

### EXCLUSIONS
- Omit greetings, thank-you statements, or introductions.
- Exclude legal disclaimers, operator prompts, procedural text, and irrelevant minutiae not affecting stock volatility.
- Entity names are prohibited in questions; include them in answers only if present in the excerpt.

--------------------------------------------
## OUTPUT FORMAT
--------------------------------------------
For each Q&A pair, use the format:

Q: <generalized, conceptual question>
A: <specific, evidence-based answer from the excerpt>
source_lines: [start_line, end_line]

If the excerpt contains no volatility-relevant content, output exactly:

[]
""".strip()



with open(BATCH_FILE, "w", encoding="utf-8") as f:

    for ecc in all_ecc_chunks:
        ecc_id = ecc["ecc_id"]

        for chunk_id, chunk in enumerate(ecc["chunks"]):

            numbered_chunk = "\n".join(
                f"{idx}: {line}" for idx, line in enumerate(chunk["text_lines"], start=1)
            )

            user_prompt = f"""
            Here is an ECC excerpt with numbered lines:

            {numbered_chunk}

            Apply the instructions from the developer message.
            Return the output in strict JSON format as specified earlier.
            If the excerpt contains nothing relevant to stock volatility, return [].
            """.strip()

            body = {
                "model": "gpt-5",
                "reasoning": {"effort": "low"},
                "instructions": template_developer_prompt,
                "input": user_prompt
            }

            entry = {
                "custom_id": f"{ecc_id}-{chunk_id}",
                "method": "POST",
                "url": "/v1/responses",
                "body": body
            }

            f.write(json.dumps(entry, ensure_ascii=False) + "\n")


print(f"Batch JSONL created: {BATCH_FILE}")

### 💲 EC QA generation (GPT-5, about 230 requests)

This step sends a request batch to the OpenAI Batch API and waits until it finishes (up to 24 hours). It needs an `OPENAI_API_KEY` (Colab secret or `.env`).

In [ ]:
# [SETUP] Windows only: the request files written with the line endings of Linux, so that the next cell uploads
# the same bytes as on Colab. On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    import glob
    _crlf, _lf = bytes([13, 10]), bytes([10])
    for _file in glob.glob("*.jsonl"):
        with open(_file, "rb") as f:
            _bytes = f.read()
        if _crlf in _bytes:
            with open(_file, "wb") as f:
                f.write(_bytes.replace(_crlf, _lf))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import json
import time
from openai import OpenAI


upload = client.files.create(
    file=open(BATCH_FILE, "rb"),
    purpose="batch"
)

file_id = upload.id
print("Uploaded batch file:", file_id)

while True:
    status = client.files.retrieve(file_id).status
    print("File status:", status)
    if status == "processed":
        break
    time.sleep(3)


batch = client.batches.create(
    input_file_id=file_id,
    endpoint="/v1/responses",
    completion_window="24h"
)

batch_id = batch.id
print("Batch job created:", batch_id)

while True:
    info = client.batches.retrieve(batch_id)
    print(
        f"Batch status: {info.status} "
        f"({info.request_counts.completed}/{info.request_counts.total})"
    )

    if info.status == "completed":
        break
    if info.status in ("failed", "expired"):
        raise RuntimeError("Batch job failed or expired.")
    time.sleep(30)



if info.request_counts.failed:
    print(f"WARNING: {info.request_counts.failed} requests failed; error file: {info.error_file_id}")
output_file_id = info.output_file_id
result_file = client.files.content(output_file_id)
result_text = result_file.text

qa_results = {}
prompt_tokens_list = {}
completion_tokens_list = {}
reasoning_tokens_list = {}
total_tokens_list = {}

for line in result_text.splitlines():
    obj = json.loads(line)
    cid = obj["custom_id"]

    response = obj["response"]["body"]

    assistant_text = None
    for item in response.get("output", []):
        if item.get("type") == "message":
            content_list = item.get("content", [])
            if content_list and "text" in content_list[0]:
                assistant_text = content_list[0]["text"].strip()
            break

    qa_results[cid] = assistant_text

    usage = response.get("usage", {})
    prompt_tokens = usage.get("input_tokens", 0)
    completion_tokens = usage.get("output_tokens", 0)
    reasoning_tokens = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)
    total_tokens = prompt_tokens + completion_tokens

    prompt_tokens_list[cid] = prompt_tokens
    completion_tokens_list[cid] = completion_tokens
    reasoning_tokens_list[cid] = reasoning_tokens
    total_tokens_list[cid] = total_tokens

    print(
        f"ID {cid}: prompt={prompt_tokens}, completion={completion_tokens}, "
        f"reasoning={reasoning_tokens}, total={total_tokens}"
    )



first_key = list(qa_results.keys())[0]
print("\nSample output for ID =", first_key)
print(qa_results[first_key])

### EC: parse the answers

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

with open("ec_kefvp_test_raw_chunks_qas_results.json", "w", encoding="utf-8") as f:
    json.dump(qa_results, f, ensure_ascii=False, indent=2)

import json

with open("ec_kefvp_test_raw_chunks_qas_results.json", "r", encoding="utf-8") as f:
    qas = json.load(f)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
total_qas = 0

ecc_qas = dict()

for qa_key, qa_val in qas.items():
    qa_list = parse_qa_value(qa_key, qa_val)
    ecc_qas[qa_key] = qa_list
    total_qas += len(qa_list)

print("Total number of QAs:", total_qas)

for k, v in ecc_qas.items():
    for qa_dict in v:
        if 'Q' not in qa_dict or 'A' not in qa_dict:
            print(f"Key: {k}, Problematic QA dictionary: {qa_dict.keys()}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
with open("ec_kefvp_test_clean_chunks_qas_results.json", "w", encoding="utf-8") as f:
    json.dump(ecc_qas, f, ensure_ascii=False, indent=2)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
for key, qas in ecc_qas.items():
    ecc_id_str, chunk_id_str = key.split("-")
    ecc_id = int(ecc_id_str)
    chunk_id = int(chunk_id_str)
    all_ecc_chunks[ecc_id]['chunks'][chunk_id]['qas'] = qas

import json

with open("ec_kefvp_test_ecc_chunks_with_qas.json", "w", encoding="utf-8") as f:
    json.dump(all_ecc_chunks, f, ensure_ascii=False, indent=2)

## 4. Splits, topics, and final files

Keeps the calls of the KeFVP train/dev/test splits (70:10:20, chronological), discovers the topics from the MAEC training questions only, assigns every question to a topic, and attaches transcript lines and audio clips.

In [ ]:
# [EXPERIMENT] The paper's code; only the download command changed (Python instead of wget, same URLs).
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec/15/maec15_train_avg_val.csv",
    "maec15_train_avg_val.csv")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec/15/maec15_dev_avg_val.csv",
    "maec15_dev_avg_val.csv")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec/15/maec15_test_avg_val.csv",
    "maec15_test_avg_val.csv")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec/16/maec16_train_avg_val.csv",
    "maec16_train_avg_val.csv")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec/16/maec16_dev_avg_val.csv",
    "maec16_dev_avg_val.csv")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec/16/maec16_test_avg_val.csv",
    "maec16_test_avg_val.csv")

import pandas as pd

maec15_train_avg_val_df = pd.read_csv('maec15_train_avg_val.csv')
maec15_test_avg_val_df = pd.read_csv('maec15_test_avg_val.csv')
maec15_dev_avg_val_df = pd.read_csv('maec15_dev_avg_val.csv')
maec16_train_avg_val_df = pd.read_csv('maec16_train_avg_val.csv')
maec16_test_avg_val_df = pd.read_csv('maec16_test_avg_val.csv')
maec16_dev_avg_val_df = pd.read_csv('maec16_dev_avg_val.csv')

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# Raw (pre-clustering, pre-label) QA pairs produced in the previous section
import json

with open("maec_clean_chunks_qas_results.json", "r", encoding="utf-8") as f:
    maec_chunks_qas = json.load(f)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import pandas as pd

# Convert transcript dates
maec_transcripts_df["date"] = pd.to_datetime(
    maec_transcripts_df["date"].astype(str),
    format="%Y%m%d"
)

# Add ECC index
maec_transcripts_df["ecc_idx"] = maec_transcripts_df.index

splits = [
    (maec15_train_avg_val_df, "kefvp_train_maec15_chunks_qas.json"),
    (maec15_test_avg_val_df,  "kefvp_test_maec15_chunks_qas.json"),
    (maec15_dev_avg_val_df,   "kefvp_dev_maec15_chunks_qas.json"),
    (maec16_train_avg_val_df, "kefvp_train_maec16_chunks_qas.json"),
    (maec16_test_avg_val_df,  "kefvp_test_maec16_chunks_qas.json"),
    (maec16_dev_avg_val_df,   "kefvp_dev_maec16_chunks_qas.json"),
]

for split_df, output_path in splits:

    split_df = split_df.copy()
    split_df["time"] = pd.to_datetime(split_df["time"])

    matched_df = maec_transcripts_df.merge(
        split_df[["ticker", "time"]],
        left_on=["ticker", "date"],
        right_on=["ticker", "time"],
        how="inner"
    ).drop(columns="time")


    ecc_idx_set = set(matched_df["ecc_idx"])

    split_chunks_qas = {
        key: value
        for key, value in maec_chunks_qas.items()
        if int(key.split("-")[0]) in ecc_idx_set
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(split_chunks_qas, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(split_chunks_qas)} entries to {output_path}")

import json

with open("kefvp_train_maec15_chunks_qas.json", "r", encoding="utf-8") as f:
    kefvp_train_maec15_chunks_qas = json.load(f)

with open("kefvp_test_maec15_chunks_qas.json", "r", encoding="utf-8") as f:
    kefvp_test_maec15_chunks_qas = json.load(f)

with open("kefvp_dev_maec15_chunks_qas.json", "r", encoding="utf-8") as f:
    kefvp_dev_maec15_chunks_qas = json.load(f)

with open("kefvp_train_maec16_chunks_qas.json", "r", encoding="utf-8") as f:
    kefvp_train_maec16_chunks_qas = json.load(f)

with open("kefvp_test_maec16_chunks_qas.json", "r", encoding="utf-8") as f:
    kefvp_test_maec16_chunks_qas = json.load(f)

with open("kefvp_dev_maec16_chunks_qas.json", "r", encoding="utf-8") as f:
    kefvp_dev_maec16_chunks_qas = json.load(f)

### Topic discovery: preliminary labels from the MAEC training questions

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
kefvp_train_all_qas = []

for chunks_qas in [kefvp_train_maec15_chunks_qas, kefvp_train_maec16_chunks_qas]:
    for key, chunk_qas in chunks_qas.items():
        ecc_number = int(key.split("-")[0])

        for qa in chunk_qas:
            kefvp_train_all_qas.append({
                "ecc_number": ecc_number,
                "Q": qa["Q"],
                "A": qa["A"]
            })

import random

random.seed(42)
random.shuffle(kefvp_train_all_qas)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import os
import json
from openai import OpenAI

template_developer_prompt = """
System: Financial Volatility Topic Extraction

# Role
Analyze earnings-call QUESTIONS to identify implied conceptual sources of financial volatility.

# Task
Given a batch of analyst or investor questions from an earnings call, extract concise topic labels representing underlying drivers of financial volatility implied by the questions.

# Instructions
- Treat the entire question batch as a single input.
- Interpret each question as signaling perceived uncertainty, risk, or variability.
- Identify recurring or significant volatility drivers implied across questions.
- Try to produce fewer, broader topic labels by selecting mechanisms that explain uncertainty raised by multiple questions.
- Focus on abstract mechanisms or processes, not resolved outcomes.
- Do NOT assume or invent answers.
- Do NOT infer drivers not directly implied by the questions.

# Labeling Rules
- Each label MUST represent one coherent conceptual source of volatility.
- Do NOT join unrelated or weakly related mechanisms in a single label.
- Avoid using conjunctions such as "and", "or", "/", unless they describe a single inseparable mechanism.
- If different mechanisms are implied, keep them as separate labels.
- Exclude company names, products, industries, geographies, and sectors.
- Exclude time periods, guidance language, and numeric details.
- Avoid near-duplicate or overlapping labels; prefer a single broader label when possible.

# Output Format (STRICT)
- Output MUST be a JSON array of strings.
- Each string is a single volatility topic label.
- Do NOT wrap labels in objects.
- Do NOT include any keys, explanations, or extra text.
- If no conceptual volatility drivers are implied, output:
[]
"""



import json
import math

BATCH_FILE = "volatility_q_batches.jsonl"
BATCH_SIZE = 100  # tune as needed

num_batches = math.ceil(len(kefvp_train_all_qas) / BATCH_SIZE)

with open(BATCH_FILE, "w", encoding="utf-8") as f:
    for batch_idx in range(num_batches):
        start = batch_idx * BATCH_SIZE
        end = start + BATCH_SIZE
        batch_qs = kefvp_train_all_qas[start:end]

        # Build numbered Question-only block
        q_block = "\n\n".join(
            f"{i+1}. {qa['Q']}"
            for i, qa in enumerate(batch_qs)
            if 'Q' in qa and qa['Q']
        )

        user_prompt = f"""
        Below is a batch of questions asked during an earnings conference call.
        Analyze the questions to identify implied sources of financial or earnings volatility.

        {q_block}

        Follow the system instructions exactly.
        Return ONLY a JSON array of strings.
        No prose. No objects. No extra text.
        """.strip()

        body = {
            "model": "gpt-5",
            "reasoning": {"effort": "low"},
            "instructions": template_developer_prompt,
            "input": user_prompt
        }

        entry = {
            "custom_id": f"q-batch-{batch_idx:05d}",
            "method": "POST",
            "url": "/v1/responses",
            "body": body
        }

        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Batch JSONL created: {BATCH_FILE}")
print(f"Total Qs: {len(kefvp_train_all_qas)}, batches: {num_batches}")

### 💲 Preliminary topic labels (GPT-5)

This step sends a request batch to the OpenAI Batch API and waits until it finishes (up to 24 hours). It needs an `OPENAI_API_KEY` (Colab secret or `.env`).

In [ ]:
# [SETUP] Windows only: the request files written with the line endings of Linux, so that the next cell uploads
# the same bytes as on Colab. On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    import glob
    _crlf, _lf = bytes([13, 10]), bytes([10])
    for _file in glob.glob("*.jsonl"):
        with open(_file, "rb") as f:
            _bytes = f.read()
        if _crlf in _bytes:
            with open(_file, "wb") as f:
                f.write(_bytes.replace(_crlf, _lf))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import json
import time
from openai import OpenAI


upload = client.files.create(
    file=open(BATCH_FILE, "rb"),
    purpose="batch"
)

file_id = upload.id
print("Uploaded batch file:", file_id)

while True:
    status = client.files.retrieve(file_id).status
    print("File status:", status)
    if status == "processed":
        break
    time.sleep(3)



batch = client.batches.create(
    input_file_id=file_id,
    endpoint="/v1/responses",
    completion_window="24h"
)

batch_id = batch.id
print("Batch job created:", batch_id)

while True:
    info = client.batches.retrieve(batch_id)
    print(
        f"Batch status: {info.status} "
        f"({info.request_counts.completed}/{info.request_counts.total})"
    )

    if info.status == "completed":
        break
    if info.status in ("failed", "expired"):
        raise RuntimeError("Batch job failed or expired.")
    time.sleep(30)


if info.request_counts.failed:
    print(f"WARNING: {info.request_counts.failed} requests failed; error file: {info.error_file_id}")
output_file_id = info.output_file_id
result_file = client.files.content(output_file_id)
result_text = result_file.text

topic_results = {}
prompt_tokens_list = {}
completion_tokens_list = {}
reasoning_tokens_list = {}
total_tokens_list = {}

for line in result_text.splitlines():
    obj = json.loads(line)
    cid = obj["custom_id"]

    response = obj["response"]["body"]

    assistant_text = None
    for item in response.get("output", []):
        if item.get("type") == "message":
            content = item.get("content", [])
            if content and "text" in content[0]:
                assistant_text = content[0]["text"].strip()
            break

    try:
        topics = json.loads(assistant_text) if assistant_text else []
    except json.JSONDecodeError:
        print(f"[WARN] {cid}: invalid JSON output")
        topics = []

    topic_results[cid] = topics

    usage = response.get("usage", {})
    prompt_tokens = usage.get("input_tokens", 0)
    completion_tokens = usage.get("output_tokens", 0)
    reasoning_tokens = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)
    total_tokens = prompt_tokens + completion_tokens

    prompt_tokens_list[cid] = prompt_tokens
    completion_tokens_list[cid] = completion_tokens
    reasoning_tokens_list[cid] = reasoning_tokens
    total_tokens_list[cid] = total_tokens

    print(
        f"ID {cid}: prompt={prompt_tokens}, completion={completion_tokens}, "
        f"reasoning={reasoning_tokens}, total={total_tokens}"
    )


first_key = list(topic_results.keys())[0]
print("\nSample output for ID =", first_key)
print(topic_results[first_key])

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
kefvp_train_labels = []

for batch_labels in topic_results.values():
    kefvp_train_labels.extend(batch_labels)

with open('kefvp_train_maec_all_potential_labels.json', 'w', encoding='utf-8') as f:
    json.dump(kefvp_train_labels, f, ensure_ascii=False, indent=2)

print(f"Saved {len(kefvp_train_labels)} labels to kefvp_train_maec_all_potential_labels.json")

### Merge the preliminary labels: remove duplicates, embed, cluster

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
kefvp_train_labels = sorted(set(kefvp_train_labels))

#### 💲 Label embeddings (text-embedding-3-large, a few cents)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import time
import math

BATCH_SIZE = 100

num_batches = math.ceil(len(kefvp_train_labels) / BATCH_SIZE)

label_embeddings = []

for batch_idx in range(num_batches):
    start = batch_idx * BATCH_SIZE
    end = start + BATCH_SIZE
    batch_labels = kefvp_train_labels[start:end]

    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=batch_labels
    )

    for label, emb_obj in zip(batch_labels, response.data):
        label_embeddings.append({
            "label": label,
            "embedding": emb_obj.embedding
        })
    time.sleep(5)


print(f"Created embeddings for {len(label_embeddings)} labels")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import numpy as np

labels = [x["label"] for x in label_embeddings]
embeddings = np.array([x["embedding"] for x in label_embeddings])

np.savez(
    "kefvp_train_maec_volatility_label_embeddings.npz",
    labels=np.array(labels, dtype=object),
    embeddings=embeddings
)

#### MiniBatch K-Means: silhouette sweep over k = 20, 25, …, 100, final model with k = 35

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from sklearn.preprocessing import normalize

X = normalize(embeddings)

In [ ]:
# [CHECK] "mini" mode: enough preliminary labels for the next cells, which try up to 100 clusters. Changes nothing.
if RUN_MODE == "mini":
    print(f"*** MINI RUN: {X.shape[0]} preliminary labels ***")
    if X.shape[0] <= 100:
        raise RuntimeError(f"only {X.shape[0]} preliminary labels, and the next cell tries up to 100 clusters. "
                           "Raise MINI_TRAIN_CALLS in the settings cell and run all cells again.")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from tqdm import tqdm

K_range = range(20, 101, 5)
sil_scores = []

for k in tqdm(K_range, desc="Evaluating k", unit="k"):
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        batch_size=512,
        n_init=10
    )
    labels_k = km.fit_predict(X)
    score = silhouette_score(X, labels_k, metric="cosine")
    sil_scores.append(score)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(K_range, sil_scores, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette score")
plt.title("Silhouette (subsampled embeddings)")
plt.show()

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
BEST_K = 35

final_km = MiniBatchKMeans(
    n_clusters=BEST_K,
    random_state=42,
    batch_size=1024,
    n_init=20
)

cluster_ids = final_km.fit_predict(X)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
clustered_labels = [
    {
        "label": label,
        "cluster": int(cid)
    }
    for label, cid in zip(labels, cluster_ids)
]

from collections import defaultdict

cluster_to_labels = defaultdict(list)

for item in clustered_labels:
    cluster_to_labels[item["cluster"]].append(item["label"])

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
template_developer_prompt = """
System role: You are an expert financial analyst specializing in earnings call risk taxonomy and volatility drivers.

Task:
You will be given a list of topic labels that all belong to ONE cluster.
These labels are noisy, overlapping, and verbose.

Your job:
- Produce ONE short cluster label that tightly represents the shared concept.
- The primary focus must be on the underlying driver of market volatility.
- The label must be concise (3–6 words).
- Use financially precise, analyst-style language.
- Words used in the label must be highly relevant to each other, semantically aligned, and appropriate as a single concept.
- Avoid conjunction bloat and unnecessary modifiers.
- Prefer abstract drivers that explain volatility rather than operational detail.

Output format (STRICT):
Return a JSON object exactly as:
{"cluster_label": "<label>"}

Do not add any explanation.
"""

import math

BATCH_FILE = "cluster_label_batches.jsonl"
BATCH_SIZE = 1

clusters = list(cluster_to_labels.items())
# clusters = list(cluster_to_labels.items())[0:5]
num_batches = math.ceil(len(clusters) / BATCH_SIZE)

with open(BATCH_FILE, "w", encoding="utf-8") as f:
    for batch_idx in range(num_batches):
        start = batch_idx * BATCH_SIZE
        end = start + BATCH_SIZE
        batch_clusters = clusters[start:end]

        for cluster_id, labels in batch_clusters:
            labels_block = "\n".join(
                f"- {lbl}" for lbl in labels if lbl
            )

            user_prompt = f"""
Topic labels:
{labels_block}

Generate a single tight cluster label.
""".strip()

            body = {
                "model": "gpt-5.2",
                "reasoning": {"effort": "medium"},
                "instructions": template_developer_prompt,
                "input": user_prompt
            }

            entry = {
                "custom_id": f"cluster-{cluster_id}",
                "method": "POST",
                "url": "/v1/responses",
                "body": body
            }

            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Batch JSONL created: {BATCH_FILE}")
print(f"Total clusters: {len(clusters)}")

### 💲 Cluster names (GPT-5.2)

This step sends a request batch to the OpenAI Batch API and waits until it finishes (up to 24 hours). It needs an `OPENAI_API_KEY` (Colab secret or `.env`).

In [ ]:
# [SETUP] Windows only: the request files written with the line endings of Linux, so that the next cell uploads
# the same bytes as on Colab. On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    import glob
    _crlf, _lf = bytes([13, 10]), bytes([10])
    for _file in glob.glob("*.jsonl"):
        with open(_file, "rb") as f:
            _bytes = f.read()
        if _crlf in _bytes:
            with open(_file, "wb") as f:
                f.write(_bytes.replace(_crlf, _lf))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import json
import time
from openai import OpenAI


upload = client.files.create(
    file=open(BATCH_FILE, "rb"),
    purpose="batch"
)

file_id = upload.id
print("Uploaded batch file:", file_id)

while True:
    status = client.files.retrieve(file_id).status
    print("File status:", status)
    if status == "processed":
        break
    time.sleep(3)




batch = client.batches.create(
    input_file_id=file_id,
    endpoint="/v1/responses",
    completion_window="24h"
)

batch_id = batch.id
print("Batch job created:", batch_id)

while True:
    info = client.batches.retrieve(batch_id)
    print(
        f"Batch status: {info.status} "
        f"({info.request_counts.completed}/{info.request_counts.total})"
    )

    if info.status == "completed":
        break
    if info.status in ("failed", "expired"):
        raise RuntimeError("Batch job failed or expired.")
    time.sleep(30)



if info.request_counts.failed:
    print(f"WARNING: {info.request_counts.failed} requests failed; error file: {info.error_file_id}")
output_file_id = info.output_file_id
result_file = client.files.content(output_file_id)
result_text = result_file.text

cluster_labels = {}
prompt_tokens_list = {}
completion_tokens_list = {}
reasoning_tokens_list = {}
total_tokens_list = {}

for line in result_text.splitlines():
    obj = json.loads(line)
    cid = obj["custom_id"]

    response = obj["response"]["body"]

    assistant_text = None
    for item in response.get("output", []):
        if item.get("type") == "message":
            content = item.get("content", [])
            if content and "text" in content[0]:
                assistant_text = content[0]["text"].strip()
            break

    try:
        parsed = json.loads(assistant_text)
        cluster_labels[cid] = parsed.get("cluster_label")
    except Exception:
        print(f"[WARN] {cid}: invalid JSON")
        cluster_labels[cid] = None

    usage = response.get("usage", {})
    prompt_tokens = usage.get("input_tokens", 0)
    completion_tokens = usage.get("output_tokens", 0)
    reasoning_tokens = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)
    total_tokens = prompt_tokens + completion_tokens

    prompt_tokens_list[cid] = prompt_tokens
    completion_tokens_list[cid] = completion_tokens
    reasoning_tokens_list[cid] = reasoning_tokens
    total_tokens_list[cid] = total_tokens

    print(
        f"ID {cid}: prompt={prompt_tokens}, completion={completion_tokens}, "
        f"reasoning={reasoning_tokens}, total={total_tokens}"
    )

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
_names = [n if n is None or isinstance(n, str) else json.dumps(n) for n in cluster_labels.values()]
if None in _names or len(set(_names)) != len(_names):
    print("WARNING: cluster names missing or duplicated; fix before classification")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
integrated_clusters = {}

for key, labels in clusters:
    cluster_key = str(key)
    label_key = f"cluster-{cluster_key}"

    integrated_clusters[cluster_key] = {
        "cluster_label": cluster_labels.get(label_key),
        "member_labels": labels
    }

import json


OUTPUT_FILE = "kefvp_train_maec_35clusters.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(integrated_clusters, f, indent=2, ensure_ascii=False)

print(f"Saved {len(integrated_clusters)} clusters to {OUTPUT_FILE}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

with open("kefvp_train_maec_35clusters.json", "r", encoding="utf-8") as f:
    integrated_clusters = json.load(f)

### Assign every question to one of the 35 topics

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
LABELS = []

for key, val in integrated_clusters.items():
    LABELS.append(val['cluster_label'])

template_developer_prompt = """
You are an expert financial NLP classifier.

Task:
- You will receive ONE question from an Earnings Call (ECC).
- You will also receive a fixed list of 35 labels.
- Assign the question to EXACTLY ONE label.
- Choose the single best semantic match.

Rules:
- Do NOT invent new labels.
- Do NOT return explanations.
- Output MUST be valid JSON.
- If multiple labels seem plausible, choose the closest primary intent.

Output format (STRICT):
{
  "label": "<ONE_LABEL_FROM_LIST>"
}
"""

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

datasets = [
    ("kefvp_train_maec15_label_batches.jsonl", kefvp_train_maec15_chunks_qas),
    ("kefvp_dev_maec15_label_batches.jsonl",   kefvp_dev_maec15_chunks_qas),
    ("kefvp_test_maec15_label_batches.jsonl",  kefvp_test_maec15_chunks_qas),
    ("kefvp_train_maec16_label_batches.jsonl", kefvp_train_maec16_chunks_qas),
    ("kefvp_dev_maec16_label_batches.jsonl",   kefvp_dev_maec16_chunks_qas),
    ("kefvp_test_maec16_label_batches.jsonl",  kefvp_test_maec16_chunks_qas),
]

for batch_file, chunks_qas in datasets:

    with open(batch_file, "w", encoding="utf-8") as f:

        for chunk_id, qas in chunks_qas.items():
            for q_idx, qa in enumerate(qas):

                question = qa["Q"]

                labels_block = "\n".join(
                    f"- {lbl}" for lbl in LABELS
                )

                user_prompt = f"""
Question:
"{question}"

Possible labels (choose ONE):
{labels_block}
""".strip()

                body = {
                    "model": "gpt-5-mini",
                    # "reasoning": {"effort": "medium"},
                    "instructions": template_developer_prompt,
                    "input": user_prompt,
                }

                entry = {
                    "custom_id": f"{chunk_id}-q{q_idx}",
                    "method": "POST",
                    "url": "/v1/responses",
                    "body": body,
                }

                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Batch JSONL created: {batch_file}")

### 💲 MAEC topic assignment (GPT-5-mini, six batches, one after another)

This step sends a request batch to the OpenAI Batch API and waits until it finishes (up to 24 hours). It needs an `OPENAI_API_KEY` (Colab secret or `.env`).

In [ ]:
# [SETUP] Windows only: the request files written with the line endings of Linux, so that the next cell uploads
# the same bytes as on Colab. On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    import glob
    _crlf, _lf = bytes([13, 10]), bytes([10])
    for _file in glob.glob("*.jsonl"):
        with open(_file, "rb") as f:
            _bytes = f.read()
        if _crlf in _bytes:
            with open(_file, "wb") as f:
                f.write(_bytes.replace(_crlf, _lf))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import json
import os
import time

batch_files = [
    "kefvp_train_maec15_label_batches.jsonl",
    "kefvp_dev_maec15_label_batches.jsonl",
    "kefvp_test_maec15_label_batches.jsonl",
    "kefvp_train_maec16_label_batches.jsonl",
    "kefvp_dev_maec16_label_batches.jsonl",
    "kefvp_test_maec16_label_batches.jsonl",
]

all_results = {}

for batch_file in batch_files:

    print(f"\n{'='*80}")
    print(batch_file)
    print(f"{'='*80}")

    upload = client.files.create(
        file=open(batch_file, "rb"),
        purpose="batch"
    )

    file_id = upload.id
    print("Uploaded:", file_id)

    while True:
        status = client.files.retrieve(file_id).status
        print("File status:", status)
        if status == "processed":
            break
        time.sleep(3)

    batch = client.batches.create(
        input_file_id=file_id,
        endpoint="/v1/responses",
        completion_window="24h"
    )

    batch_id = batch.id
    print("Batch:", batch_id)

    while True:
        info = client.batches.retrieve(batch_id)

        print(
            f"Status: {info.status} "
            f"({info.request_counts.completed}/{info.request_counts.total})"
        )

        if info.status == "completed":
            break

        if info.status in ("failed", "expired", "cancelled"):
            raise RuntimeError(f"{batch_file} failed.")

        time.sleep(30)

    if info.request_counts.failed:
        print(f"WARNING: {info.request_counts.failed} requests failed; error file: {info.error_file_id}")
    output_file_id = info.output_file_id
    result_text = client.files.content(output_file_id).text

    prompt_tokens_list = {}
    completion_tokens_list = {}
    reasoning_tokens_list = {}
    total_tokens_list = {}
    qa_labels = {}

    for line in result_text.splitlines():

        obj = json.loads(line)
        cid = obj["custom_id"]
        response = obj["response"]["body"]

        assistant_text = None
        for item in response.get("output", []):
            if item.get("type") == "message":
                content = item.get("content", [])
                if content and "text" in content[0]:
                    assistant_text = content[0]["text"].strip()
                break

        try:
            parsed = json.loads(assistant_text)
            qa_labels[cid] = parsed["label"]
        except Exception:
            print(f"[WARN] {cid}: invalid JSON")
            qa_labels[cid] = None

        usage = response.get("usage", {})

        prompt = usage.get("input_tokens", 0)
        completion = usage.get("output_tokens", 0)
        reasoning = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)
        total = prompt + completion

        prompt_tokens_list[cid] = prompt
        completion_tokens_list[cid] = completion
        reasoning_tokens_list[cid] = reasoning
        total_tokens_list[cid] = total

    all_results[batch_file] = {
        "batch_id": batch_id,
        "file_id": file_id,
        "qa_labels": qa_labels,
        "prompt_tokens": prompt_tokens_list,
        "completion_tokens": completion_tokens_list,
        "reasoning_tokens": reasoning_tokens_list,
        "total_tokens": total_tokens_list,
    }

    print(f"Finished {batch_file}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

output_dir = ""

save_map = {
    "kefvp_train_maec15_label_batches.jsonl":
        "kefvp_train_maec15_qa_raw_final_label.json",

    "kefvp_dev_maec15_label_batches.jsonl":
        "kefvp_dev_maec15_qa_raw_final_label.json",

    "kefvp_test_maec15_label_batches.jsonl":
        "kefvp_test_maec15_qa_raw_final_label.json",

    "kefvp_train_maec16_label_batches.jsonl":
        "kefvp_train_maec16_qa_raw_final_label.json",

    "kefvp_dev_maec16_label_batches.jsonl":
        "kefvp_dev_maec16_qa_raw_final_label.json",

    "kefvp_test_maec16_label_batches.jsonl":
        "kefvp_test_maec16_qa_raw_final_label.json",

}

for batch_file, result in all_results.items():

    if batch_file not in save_map:
        continue

    out_name = save_map[batch_file]
    out_path = os.path.join(output_dir, out_name)

    qa_labels = result["qa_labels"]

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(qa_labels, f, ensure_ascii=False, indent=2)

    print(f"Saved: {out_path}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# Reload the per-split classification results and sanity-check that every
# predicted label is one of the 35 known cluster labels.
import json

label_files = {
    "kefvp_train_maec15_chunks_qas": "kefvp_train_maec15_qa_raw_final_label.json",
    "kefvp_dev_maec15_chunks_qas": "kefvp_dev_maec15_qa_raw_final_label.json",
    "kefvp_test_maec15_chunks_qas": "kefvp_test_maec15_qa_raw_final_label.json",
    "kefvp_train_maec16_chunks_qas": "kefvp_train_maec16_qa_raw_final_label.json",
    "kefvp_dev_maec16_chunks_qas": "kefvp_dev_maec16_qa_raw_final_label.json",
    "kefvp_test_maec16_chunks_qas": "kefvp_test_maec16_qa_raw_final_label.json",
}

qa_raw_final_labels = {}

for var_name, filename in label_files.items():
    path = f"{filename}"
    with open(path, "r", encoding="utf-8") as f:
        labels_dict = json.load(f)
    qa_raw_final_labels[var_name] = labels_dict

    invalid = {qid: lbl for qid, lbl in labels_dict.items() if lbl not in LABELS}
    print(f"{var_name}: {len(labels_dict) - len(invalid)} valid / {len(invalid)} invalid labels")
    for qid, lbl in invalid.items():
        print(f"  [WARN] {qid}: {lbl!r} not in LABELS")

kefvp_train_maec15_qa_raw_final_label = qa_raw_final_labels["kefvp_train_maec15_chunks_qas"]
kefvp_dev_maec15_qa_raw_final_label = qa_raw_final_labels["kefvp_dev_maec15_chunks_qas"]
kefvp_test_maec15_qa_raw_final_label = qa_raw_final_labels["kefvp_test_maec15_chunks_qas"]
kefvp_train_maec16_qa_raw_final_label = qa_raw_final_labels["kefvp_train_maec16_chunks_qas"]
kefvp_dev_maec16_qa_raw_final_label = qa_raw_final_labels["kefvp_dev_maec16_chunks_qas"]
kefvp_test_maec16_qa_raw_final_label = qa_raw_final_labels["kefvp_test_maec16_chunks_qas"]

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
for chunk_id, qas in kefvp_train_maec15_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_train_maec15_qa_raw_final_label.get(qid)

for chunk_id, qas in kefvp_dev_maec15_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_dev_maec15_qa_raw_final_label.get(qid)

for chunk_id, qas in kefvp_test_maec15_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_test_maec15_qa_raw_final_label.get(qid)


for chunk_id, qas in kefvp_train_maec16_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_train_maec16_qa_raw_final_label.get(qid)

for chunk_id, qas in kefvp_dev_maec16_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_dev_maec16_qa_raw_final_label.get(qid)

for chunk_id, qas in kefvp_test_maec16_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_test_maec16_qa_raw_final_label.get(qid)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

OUTPUT_DIR = ""

datasets = {
    "kefvp_train_maec15_chunks_qas_with_final_label.json": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15_chunks_qas_with_final_label.json": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15_chunks_qas_with_final_label.json": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16_chunks_qas_with_final_label.json": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16_chunks_qas_with_final_label.json": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16_chunks_qas_with_final_label.json": kefvp_test_maec16_chunks_qas,
}

for filename, data in datasets.items():
    output_file = os.path.join(OUTPUT_DIR, filename)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} chunks to {output_file}")

#### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

with open("ec_kefvp_test_clean_chunks_qas_results.json", "r", encoding="utf-8") as f:
    kefvp_test_ec_chunks_qas = json.load(f)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json

datasets = [
    ("kefvp_test_ec_label_batches.jsonl", kefvp_test_ec_chunks_qas),
]

for batch_file, chunks_qas in datasets:

    with open(batch_file, "w", encoding="utf-8") as f:

        for chunk_id, qas in chunks_qas.items():
            for q_idx, qa in enumerate(qas):

                question = qa["Q"]

                labels_block = "\n".join(
                    f"- {lbl}" for lbl in LABELS
                )

                user_prompt = f"""
Question:
"{question}"

Possible labels (choose ONE):
{labels_block}
""".strip()

                body = {
                    "model": "gpt-5-mini",
                    # "reasoning": {"effort": "medium"},
                    "instructions": template_developer_prompt,
                    "input": user_prompt,
                }

                entry = {
                    "custom_id": f"{chunk_id}-q{q_idx}",
                    "method": "POST",
                    "url": "/v1/responses",
                    "body": body,
                }

                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Batch JSONL created: {batch_file}")

### 💲 EC topic assignment (GPT-5-mini)

This step sends a request batch to the OpenAI Batch API and waits until it finishes (up to 24 hours). It needs an `OPENAI_API_KEY` (Colab secret or `.env`).

In [ ]:
# [SETUP] Windows only: the request files written with the line endings of Linux, so that the next cell uploads
# the same bytes as on Colab. On Linux and on Colab this cell changes nothing.
if os.name == "nt":
    import glob
    _crlf, _lf = bytes([13, 10]), bytes([10])
    for _file in glob.glob("*.jsonl"):
        with open(_file, "rb") as f:
            _bytes = f.read()
        if _crlf in _bytes:
            with open(_file, "wb") as f:
                f.write(_bytes.replace(_crlf, _lf))

In [ ]:
# [EXPERIMENT] The paper's code, unchanged. PAID: this cell sends requests to the OpenAI API.
import json
import os
import time

batch_files = [
    "kefvp_test_ec_label_batches.jsonl",
]

all_results = {}

for batch_file in batch_files:

    print(f"\n{'='*80}")
    print(batch_file)
    print(f"{'='*80}")

    upload = client.files.create(
        file=open(batch_file, "rb"),
        purpose="batch"
    )

    file_id = upload.id
    print("Uploaded:", file_id)

    while True:
        status = client.files.retrieve(file_id).status
        print("File status:", status)
        if status == "processed":
            break
        time.sleep(3)

    batch = client.batches.create(
        input_file_id=file_id,
        endpoint="/v1/responses",
        completion_window="24h"
    )

    batch_id = batch.id
    print("Batch:", batch_id)

    while True:
        info = client.batches.retrieve(batch_id)

        print(
            f"Status: {info.status} "
            f"({info.request_counts.completed}/{info.request_counts.total})"
        )

        if info.status == "completed":
            break

        if info.status in ("failed", "expired", "cancelled"):
            raise RuntimeError(f"{batch_file} failed.")

        time.sleep(30)

    if info.request_counts.failed:
        print(f"WARNING: {info.request_counts.failed} requests failed; error file: {info.error_file_id}")
    output_file_id = info.output_file_id
    result_text = client.files.content(output_file_id).text

    prompt_tokens_list = {}
    completion_tokens_list = {}
    reasoning_tokens_list = {}
    total_tokens_list = {}
    qa_labels = {}

    for line in result_text.splitlines():

        obj = json.loads(line)
        cid = obj["custom_id"]
        response = obj["response"]["body"]

        assistant_text = None
        for item in response.get("output", []):
            if item.get("type") == "message":
                content = item.get("content", [])
                if content and "text" in content[0]:
                    assistant_text = content[0]["text"].strip()
                break

        try:
            parsed = json.loads(assistant_text)
            qa_labels[cid] = parsed["label"]
        except Exception:
            print(f"[WARN] {cid}: invalid JSON")
            qa_labels[cid] = None

        usage = response.get("usage", {})

        prompt = usage.get("input_tokens", 0)
        completion = usage.get("output_tokens", 0)
        reasoning = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)
        total = prompt + completion

        prompt_tokens_list[cid] = prompt
        completion_tokens_list[cid] = completion
        reasoning_tokens_list[cid] = reasoning
        total_tokens_list[cid] = total

    all_results[batch_file] = {
        "batch_id": batch_id,
        "file_id": file_id,
        "qa_labels": qa_labels,
        "prompt_tokens": prompt_tokens_list,
        "completion_tokens": completion_tokens_list,
        "reasoning_tokens": reasoning_tokens_list,
        "total_tokens": total_tokens_list,
    }

    print(f"Finished {batch_file}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

output_dir = ""

save_map = {
    "kefvp_test_ec_label_batches.jsonl":
        "kefvp_test_ec_qa_raw_final_label.json",
}

for batch_file, result in all_results.items():

    if batch_file not in save_map:
        continue

    out_name = save_map[batch_file]
    out_path = os.path.join(output_dir, out_name)

    qa_labels = result["qa_labels"]

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(qa_labels, f, ensure_ascii=False, indent=2)

    print(f"Saved: {out_path}")

import json

with open("kefvp_test_ec_qa_raw_final_label.json", "r", encoding="utf-8") as f:
    kefvp_test_ec_qa_raw_final_label = json.load(f)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
invalid_labels = {}
valid_count = 0

for qid, label in kefvp_test_ec_qa_raw_final_label.items():
    if label in LABELS:
        valid_count += 1
    else:
        invalid_labels[qid] = label

print(f"Valid labels: {valid_count}")
print(f"Invalid labels: {len(invalid_labels)}")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
for chunk_id, qas in kefvp_test_ec_chunks_qas.items():
    for q_idx, qa in enumerate(qas):
        qid = f"{chunk_id}-q{q_idx}"
        qa["topic"] = kefvp_test_ec_qa_raw_final_label.get(qid)

import json
import os

OUTPUT_DIR = ""

datasets = {
    "kefvp_test_ec_chunks_qas_with_final_label.json": kefvp_test_ec_chunks_qas,
}

for filename, data in datasets.items():
    output_file = os.path.join(OUTPUT_DIR, filename)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} chunks to {output_file}")

### Attach transcript lines and audio clips (chunks of about 100 lines, as above) and write the final files

#### MAEC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# split_ecc_into_chunks is defined in generate question section
maec_all_ecc_chunks = []

for idx, row in maec_transcripts_df.iterrows():
    transcript = row["annotated_transcript_text"]
    turn_audio_map = row["turn_audio_map"]

    chunks = split_ecc_into_chunks(transcript, turn_audio_map, max_lines=100)
    maec_all_ecc_chunks.append({
        "ecc_id": idx,
        "chunks": chunks
    })

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
datasets = {
    "kefvp_train_maec15_chunks_qas": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15_chunks_qas": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15_chunks_qas": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16_chunks_qas": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16_chunks_qas": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16_chunks_qas": kefvp_test_maec16_chunks_qas,
}

for name, dataset in datasets.items():
    new_dataset = {}

    for key, qas in dataset.items():
        ecc_id_str, chunk_id_str = key.split("-")
        ecc_id = int(ecc_id_str)
        chunk_id = int(chunk_id_str)

        chunk_data = maec_all_ecc_chunks[ecc_id]["chunks"][chunk_id]

        new_dataset[key] = {
            "qas": qas,
            "text_lines": chunk_data.get("text_lines"),
            "audio_map": chunk_data.get("audio_map"),
            "audio_path": maec_transcripts_df.iloc[ecc_id]["audio_path"],
        }

    datasets[name] = new_dataset

kefvp_train_maec15_chunks_qas = datasets["kefvp_train_maec15_chunks_qas"]
kefvp_dev_maec15_chunks_qas = datasets["kefvp_dev_maec15_chunks_qas"]
kefvp_test_maec15_chunks_qas = datasets["kefvp_test_maec15_chunks_qas"]
kefvp_train_maec16_chunks_qas = datasets["kefvp_train_maec16_chunks_qas"]
kefvp_dev_maec16_chunks_qas = datasets["kefvp_dev_maec16_chunks_qas"]
kefvp_test_maec16_chunks_qas = datasets["kefvp_test_maec16_chunks_qas"]

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

OUTPUT_DIR = QA_DIR

datasets = {
    "kefvp_train_maec15_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_train_maec15_chunks_qas,
    "kefvp_dev_maec15_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_dev_maec15_chunks_qas,
    "kefvp_test_maec15_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_test_maec15_chunks_qas,
    "kefvp_train_maec16_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_train_maec16_chunks_qas,
    "kefvp_dev_maec16_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_dev_maec16_chunks_qas,
    "kefvp_test_maec16_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_test_maec16_chunks_qas,
}

for filename, data in datasets.items():
    output_file = os.path.join(OUTPUT_DIR, filename)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} chunks to {output_file}")

#### EC

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# split_ecc_into_chunks is defined in generate question section
ec_all_ecc_chunks = []

for idx, row in ec_transcripts_test_df.iterrows():
    transcript = row["annotated_transcript_text"]
    turn_audio_map = row["turn_audio_map"]

    chunks = split_ecc_into_chunks(transcript, turn_audio_map, max_lines=100)
    ec_all_ecc_chunks.append({
        "ecc_id": idx,
        "chunks": chunks
    })

datasets = {
    "kefvp_test_ec_chunks_qas": kefvp_test_ec_chunks_qas,
}

for name, dataset in datasets.items():
    new_dataset = {}

    for key, qas in dataset.items():
        ecc_id_str, chunk_id_str = key.split("-")
        ecc_id = int(ecc_id_str)
        chunk_id = int(chunk_id_str)

        chunk_data = ec_all_ecc_chunks[ecc_id]["chunks"][chunk_id]

        new_dataset[key] = {
            "qas": qas,
            "text_lines": chunk_data.get("text_lines"),
            "audio_map": chunk_data.get("audio_map"),
            "audio_path": ec_transcripts_test_df.iloc[ecc_id]["audio_path"],
        }

    datasets[name] = new_dataset

kefvp_test_ec_chunks_qas = datasets["kefvp_test_ec_chunks_qas"]

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

OUTPUT_DIR = QA_DIR

datasets = {
    "kefvp_test_ec_chunks_qas_with_final_label_with_textlines_with_audiomap.json": kefvp_test_ec_chunks_qas,
}

for filename, data in datasets.items():
    output_file = os.path.join(OUTPUT_DIR, filename)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} chunks to {output_file}")

## 5. Call map

Ticker, date and call folder of every call id. Notebook 2 uses it to attach `ticker` and `date` to each embedding, which notebook 3 uses to join embeddings with the volatility labels. The topic list is copied next to the QA files.

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import json
import os

maec_call_map = {
    str(idx): {
        "ticker": row["ticker"],
        "date": os.path.basename(os.path.dirname(row["path"])).split("_", 1)[0],   # YYYYMMDD from the folder name
        "text_file_name": os.path.basename(os.path.dirname(row["path"])),
    }
    for idx, row in maec_transcripts_df.iterrows()
}

ec_keys = ec_test_avg_val_df[["ticker", "name", "date", "text_file_name"]].rename(columns={"name": "company_name"})
ec_map_df = (
    ec_transcripts_test_df[["company_name", "date"]]
    .reset_index()
    .merge(ec_keys, on=["company_name", "date"], how="left")
)
assert len(ec_map_df) == len(ec_transcripts_test_df) and ec_map_df["ticker"].notna().all()
ec_call_map = {
    str(r["index"]): {"ticker": r["ticker"], "date": r["date"], "text_file_name": r["text_file_name"]}
    for _, r in ec_map_df.iterrows()
}

with open(os.path.join(QA_DIR, "call_map.json"), "w", encoding="utf-8") as f:
    json.dump({"maec": maec_call_map, "ec": ec_call_map}, f, ensure_ascii=False, indent=1)
print(f"call map: {len(maec_call_map)} MAEC calls, {len(ec_call_map)} EC test calls")

import shutil
shutil.copy2("kefvp_train_maec_35clusters.json", os.path.join(QA_DIR, "kefvp_train_maec_35clusters.json"))

In [ ]:
# [CHECK] The files this run wrote. Changes nothing.
_files = [f"kefvp_{s}_chunks_qas_with_final_label_with_textlines_with_audiomap.json"
          for s in ("train_maec15", "dev_maec15", "test_maec15", "train_maec16", "dev_maec16", "test_maec16", "test_ec")]
_missing = [n for n in _files + ["call_map.json", "kefvp_train_maec_35clusters.json"]
            if not os.path.exists(os.path.join(QA_DIR, n))]
for _name in _files:
    if _name not in _missing:
        with open(os.path.join(QA_DIR, _name), encoding="utf-8") as f:
            _d = json.load(f)
        print(f"  {_name.split('_chunks')[0]:20s} {len(_d):5d} chunks | {sum(len(c['qas']) for c in _d.values()):6d} QA units")
if _missing:
    raise FileNotFoundError(f"{len(_missing)} of the 9 output files are not in {QA_DIR}: {_missing}")
print(f"\nAll 9 files are in {QA_DIR}.")
if RUN_MODE == "mini":
    print("MINI RUN PASSED: every cell ran, including the paid ones. These QA units are a test output.")
    print('To test notebook 2 on them: set QA_DIR in its settings cell to this folder, with RUN_MODE = "smoke".')
    print('Next here: set RUN_MODE = "full" in the settings cell and run all cells again.')
else:
    print("Next: notebook 2, with QA_DIR in its settings cell set to this folder (and the same AUDIO_DIR).")